In [ ]:
"""
Sell Bot v2 — Smart Position Manager
═════════════════════════════════════
Monitors open positions via IBKR + MongoDB.

EXIT STRATEGY (designed for momentum breakout plays):
─────────────────────────────────────────────────────
Phase 1 — PROTECTION (from entry until profitable):
  • Hard stop: -10% from entry (expanded from 3% to let breakouts breathe)
  • Flash crash: -5% drop in a single 30-second interval → emergency exit

Phase 2 — PARTIAL PROFIT at +50%:
  • Sell 50% of position, lock in gains
  • Remaining 50% enters "runner" mode with trailing stop

Phase 3 — RUNNER (remaining shares after partial):
  • Trailing stop tightens in tiers based on how far price has run:
      +50% to +75%  →  trail 15% from high  (give it room to consolidate)
      +75% to +100% →  trail 12% from high
      +100% to +150% → trail 10% from high
      +150%+         → trail  8% from high  (tightest — protect big win)
  • PSAR flip bearish → soft exit (only triggers if also below EMA 9)
  • EMA 9 crosses below EMA 21 → exit runner (trend is done)

Phase 4 — TIME DECAY (if stock goes nowhere):
  • If position is between -5% and +10% after 2 hours → exit
    (momentum play didn't work, free up capital)

EMERGENCY EXITS (always active, any phase):
  • Flash crash detection (configurable threshold)
  • Liquidity crisis (spread > 0.5%)
  • These use aggressive limit → market order fallback
"""

import logging
import sys
import time
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta
import random
from typing import Optional
from dataclasses import dataclass

from pymongo import MongoClient
from ib_insync import *

import warnings
warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# ─── IBKR ─────────────────────────────────────────────────────────────────────
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)

# ─── Exit Thresholds ──────────────────────────────────────────────────────────
HARD_STOP_PCT          = 0.10    # 10% max loss from entry (expanded from 3%)
FLASH_CRASH_PCT        = 0.05    # 5% drop in one interval → emergency
LIQUIDITY_SPREAD_PCT   = 0.005   # 0.5% spread → liquidity crisis

# ─── Partial Profit ───────────────────────────────────────────────────────────
PARTIAL_PROFIT_PCT     = 0.50    # Take partial at +50%
PARTIAL_SELL_FRACTION  = 0.50    # Sell 50% of position

# ─── Tiered Trailing Stops (for runner after partial) ─────────────────────────
# Each tuple: (profit_threshold, trail_pct_from_high)
# Evaluated top-down — first match wins
TRAILING_TIERS = [
    (1.50, 0.08),   # +150%+ profit → trail 8% from high
    (1.00, 0.10),   # +100%+ profit → trail 10%
    (0.75, 0.12),   # +75%+  profit → trail 12%
    (0.50, 0.15),   # +50%+  profit → trail 15%
]
DEFAULT_TRAILING_PCT   = 0.20    # Before +50%, if somehow needed: 20%

# ─── Time Decay ───────────────────────────────────────────────────────────────
STALE_TRADE_HOURS      = 2      # Exit if position stagnates after this long
STALE_TRADE_MIN_PNL    = -0.05  # … and P&L is between this …
STALE_TRADE_MAX_PNL    = 0.10   # … and this (going nowhere)

# ─── Technical Exit Signals ───────────────────────────────────────────────────
# PSAR + EMA crossover used as confirmation for runner exits
EMA_FAST_PERIOD        = 9
EMA_SLOW_PERIOD        = 21

# ─── Monitoring ───────────────────────────────────────────────────────────────
MONITOR_INTERVAL       = 30     # seconds between checks
EMERGENCY_TIMEOUT      = 5      # seconds to wait for limit fill before market

# ─── MongoDB ──────────────────────────────────────────────────────────────────
mongo_client     = MongoClient("mongodb://localhost:27017/")
db               = mongo_client["breakout_db"]
trades_collection = db["trades_v2"]


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING
# ══════════════════════════════════════════════════════════════════════════════

class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh  = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh  = logging.FileHandler("sell_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("sell_bot")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


# ══════════════════════════════════════════════════════════════════════════════
# IBKR CLIENT
# ══════════════════════════════════════════════════════════════════════════════

class IBKRSellClient:
    def __init__(self):
        util.startLoop()
        self.ib = IB()

    def connect(self):
        self.ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
        log.info(f"IBKR connected | accounts: {self.ib.wrapper.accounts}")

    def disconnect(self):
        self.ib.disconnect()

    # ── Price helpers ─────────────────────────────────────────────────────────

    def get_last_price(self, symbol: str) -> Optional[float]:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "106", False, False)
        self.ib.sleep(3)
        self.ib.cancelMktData(contract)

        for label, val in [("last", ticker.last), ("ask", ticker.ask),
                           ("bid", ticker.bid), ("close", ticker.close)]:
            if val and float(val) > 0:
                log.debug(f"{symbol}: price source='{label}' ${float(val):.4f}")
                return float(val)

        log.warning(f"{symbol}: no price available")
        return None

    def get_ticker(self, symbol: str) -> Ticker:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "", False, False)
        self.ib.sleep(2)
        return ticker

    # ── Historical bars ───────────────────────────────────────────────────────

    def get_bars(self, symbol: str, n: int = 70) -> Optional[pd.DataFrame]:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)

        for what_to_show in ("TRADES", "MIDPOINT"):
            raw = self.ib.reqHistoricalData(
                contract,
                endDateTime="",
                durationStr="10 D",
                barSizeSetting="5 mins",
                whatToShow=what_to_show,
                useRTH=False,
                keepUpToDate=False,
            )
            if raw:
                df = pd.DataFrame([
                    {"open": b.open, "high": b.high, "low": b.low,
                     "close": b.close, "volume": getattr(b, "volume", 0)}
                    for b in raw
                ])
                return df.tail(n).reset_index(drop=True)

        log.warning(f"{symbol}: no bars returned")
        return None

    # ── Technical indicators ──────────────────────────────────────────────────

    def compute_exit_signals(self, symbol: str) -> dict:
        """
        Compute PSAR, EMA 9, EMA 21 for exit decisions.
        Returns dict with signal flags or empty dict on failure.
        """
        df = self.get_bars(symbol, n=70)
        if df is None or len(df) < 25:
            log.warning(f"{symbol}: not enough bars for exit signals")
            return {}

        # Parabolic SAR
        psar_vals = self._compute_psar(df["high"].values, df["low"].values,
                                       df["close"].values)

        # EMA 9 / 21
        ema_fast = df["close"].ewm(span=EMA_FAST_PERIOD, adjust=False).mean()
        ema_slow = df["close"].ewm(span=EMA_SLOW_PERIOD, adjust=False).mean()

        price     = float(df["close"].iloc[-1])
        psar_now  = float(psar_vals[-1])
        ema9_now  = float(ema_fast.iloc[-1])
        ema21_now = float(ema_slow.iloc[-1])

        # Previous bar for crossover detection
        ema9_prev  = float(ema_fast.iloc[-2])
        ema21_prev = float(ema_slow.iloc[-2])

        psar_bearish       = psar_now > price       # SAR above price = downtrend
        ema_death_cross    = (ema9_prev >= ema21_prev) and (ema9_now < ema21_now)
        ema_bearish        = ema9_now < ema21_now    # EMA 9 below 21
        price_below_ema9   = price < ema9_now

        signals = {
            "price":             price,
            "psar":              psar_now,
            "ema_9":             ema9_now,
            "ema_21":            ema21_now,
            "psar_bearish":      psar_bearish,
            "ema_death_cross":   ema_death_cross,
            "ema_bearish":       ema_bearish,
            "price_below_ema9":  price_below_ema9,
        }

        log.info(
            f"{symbol} signals: PSAR={psar_now:.4f}({'BEAR' if psar_bearish else 'BULL'}) "
            f"EMA9={ema9_now:.4f} EMA21={ema21_now:.4f} "
            f"cross={'YES' if ema_death_cross else 'no'} "
            f"below9={'YES' if price_below_ema9 else 'no'}"
        )
        return signals

    @staticmethod
    def _compute_psar(highs, lows, closes, iaf=0.02, max_af=0.2) -> np.ndarray:
        n    = len(closes)
        psar = closes.copy().astype(float)
        bull = True
        af   = iaf
        hp   = float(highs[0])
        lp   = float(lows[0])

        for i in range(2, n):
            if bull:
                psar[i] = psar[i - 1] + af * (hp - psar[i - 1])
                psar[i] = min(psar[i], lows[i - 1], lows[i - 2])
                if lows[i] < psar[i]:
                    bull    = False
                    psar[i] = hp
                    lp      = lows[i]
                    af      = iaf
                else:
                    if highs[i] > hp:
                        hp = highs[i]
                        af = min(af + iaf, max_af)
            else:
                psar[i] = psar[i - 1] + af * (lp - psar[i - 1])
                psar[i] = max(psar[i], highs[i - 1], highs[i - 2])
                if highs[i] > psar[i]:
                    bull    = True
                    psar[i] = lp
                    hp      = highs[i]
                    af      = iaf
                else:
                    if lows[i] < lp:
                        lp = lows[i]
                        af = min(af + iaf, max_af)
        return psar

    # ── Order execution ───────────────────────────────────────────────────────

    def get_avg_fill_price_from_broker(self, symbol: str, order_id: int) -> Optional[float]:
        for trade in self.ib.trades():
            if (trade.contract.symbol == symbol
                    and trade.order.orderId == order_id
                    and trade.orderStatus.avgFillPrice > 0):
                return float(trade.orderStatus.avgFillPrice)

        fills   = self.ib.fills()
        matched = [f for f in fills
                   if f.contract.symbol == symbol and f.execution.orderId == order_id]
        if matched:
            total_qty  = sum(f.execution.shares for f in matched)
            total_cost = sum(f.execution.shares * f.execution.price for f in matched)
            return total_cost / total_qty

        return None

    def place_limit_sell(self, symbol: str, price: float, qty: int,
                         emergency: bool = False) -> Trade:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)

        if emergency:
            price = price * 0.99  # 1% below for faster fill

        order = LimitOrder("SELL", qty, round(price, 2))
        order.outsideRth = True
        trade = self.ib.placeOrder(contract, order)
        self.ib.sleep(1)

        log.info(f"SELL LIMIT {symbol} x{qty} @ ${price:.2f} "
                 f"orderId={trade.order.orderId} emergency={emergency}")
        return trade

    def place_market_sell(self, symbol: str, qty: int) -> Trade:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        order = MarketOrder("SELL", qty)
        trade = self.ib.placeOrder(contract, order)
        self.ib.sleep(1)
        log.info(f"SELL MARKET {symbol} x{qty} orderId={trade.order.orderId}")
        return trade

    def get_sell_fill_price(self, symbol: str, sell_order_id: int,
                            fallback_price: float) -> float:
        self.ib.sleep(3)
        avg = self.get_avg_fill_price_from_broker(symbol, sell_order_id)
        if avg:
            return avg
        log.warning(f"{symbol}: sell fill not confirmed, using fallback ${fallback_price:.4f}")
        return fallback_price


# ══════════════════════════════════════════════════════════════════════════════
# MONGODB HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def get_open_trades() -> list:
    return list(trades_collection.find({"status": "open"}))


def get_effective_entry_price(trade: dict, ibkr: IBKRSellClient) -> float:
    """Priority: avg_fill_price (DB) → broker lookup → entry_price (DB)."""
    if trade.get("avg_fill_price"):
        return float(trade["avg_fill_price"])

    order_id = trade.get("order_id")
    symbol   = trade.get("instrument") or trade.get("symbol")

    if order_id:
        broker_avg = ibkr.get_avg_fill_price_from_broker(symbol, order_id)
        if broker_avg:
            trades_collection.update_one(
                {"_id": trade["_id"]},
                {"$set": {"avg_fill_price": broker_avg, "entry_price": broker_avg}},
            )
            return broker_avg

    return float(trade["entry_price"])


def update_trade_field(trade_id, updates: dict):
    updates["updated_at"] = datetime.now(timezone.utc)
    trades_collection.update_one({"_id": trade_id}, {"$set": updates})


def close_trade(trade: dict, exit_price: float, reason: str, qty_sold: int = None):
    """Close (or partially close) a trade in MongoDB."""
    symbol      = trade.get("instrument") or trade.get("symbol")
    entry_price = float(trade.get("avg_fill_price") or trade["entry_price"])
    total_qty   = trade.get("quantity") or trade.get("qty")
    pnl_pct     = (exit_price - entry_price) / entry_price

    if qty_sold is None:
        qty_sold = total_qty

    realized_pnl = (exit_price - entry_price) * qty_sold

    if qty_sold >= total_qty:
        # Full close
        trades_collection.update_one(
            {"_id": trade["_id"]},
            {"$set": {
                "status":           "closed",
                "exit_price":       exit_price,
                "exit_timestamp":   datetime.now(timezone.utc),
                "reason":           reason,
                "pnl_pct":          round(pnl_pct * 100, 2),
                "realized_pnl":     realized_pnl,
                "updated_at":       datetime.now(timezone.utc),
            }},
        )
        log.info(
            f"CLOSED {symbol} | entry=${entry_price:.4f} exit=${exit_price:.4f} "
            f"pnl={pnl_pct * 100:+.2f}% | ${realized_pnl:+.2f} | reason={reason}"
        )
    else:
        # Partial close — reduce quantity, record partial exit
        remaining = total_qty - qty_sold
        trades_collection.update_one(
            {"_id": trade["_id"]},
            {"$set": {
                "quantity":              remaining,
                "partial_exit_price":    exit_price,
                "partial_exit_qty":      qty_sold,
                "partial_exit_time":     datetime.now(timezone.utc),
                "partial_exit_reason":   reason,
                "partial_realized_pnl":  realized_pnl,
                "phase":                 "runner",
                "updated_at":            datetime.now(timezone.utc),
            }},
        )
        log.info(
            f"PARTIAL CLOSE {symbol} | sold {qty_sold}/{total_qty} @ ${exit_price:.4f} "
            f"pnl={pnl_pct * 100:+.2f}% | ${realized_pnl:+.2f} | "
            f"remaining={remaining} shares → runner mode"
        )


# ══════════════════════════════════════════════════════════════════════════════
# RISK MANAGEMENT
# ══════════════════════════════════════════════════════════════════════════════

def get_trailing_stop_pct(pnl_pct: float) -> float:
    """
    Return the trailing stop percentage based on how far price has run.
    Tighter trail as profit grows (protect bigger wins).
    """
    for threshold, trail in TRAILING_TIERS:
        if pnl_pct >= threshold:
            return trail
    return DEFAULT_TRAILING_PCT


def compute_trailing_stop(entry_price: float, highest_price: float,
                          current_pnl_pct: float) -> float:
    """Compute the effective trailing stop price."""
    trail_pct  = get_trailing_stop_pct(current_pnl_pct)
    trail_stop = highest_price * (1 - trail_pct)
    hard_stop  = entry_price * (1 - HARD_STOP_PCT)
    return max(trail_stop, hard_stop)


def check_flash_crash(symbol: str, current_price: float, trade: dict) -> Optional[str]:
    """Detect sudden price drops within a single monitoring interval."""
    last_price = trade.get("last_price")

    # Always update for next iteration
    update_trade_field(trade["_id"], {
        "last_price":      current_price,
        "last_price_time": datetime.now(timezone.utc),
    })

    if last_price and last_price > 0:
        drop_pct = (current_price - last_price) / last_price
        if drop_pct <= -FLASH_CRASH_PCT:
            log.warning(f"{symbol}: FLASH CRASH {drop_pct * 100:.2f}% in {MONITOR_INTERVAL}s")
            return f"flash_crash_{abs(drop_pct) * 100:.1f}pct"

    return None


def check_liquidity_crisis(ibkr: IBKRSellClient, symbol: str) -> bool:
    """Detect bid-ask spread blowing out."""
    try:
        ticker = ibkr.get_ticker(symbol)
        if ticker.bid and ticker.ask and ticker.bid > 0:
            mid        = (ticker.ask + ticker.bid) / 2
            spread_pct = (ticker.ask - ticker.bid) / mid
            if spread_pct > LIQUIDITY_SPREAD_PCT:
                log.warning(f"{symbol}: liquidity crisis — spread {spread_pct * 100:.2f}%")
                return True
    except Exception as e:
        log.error(f"{symbol}: liquidity check error — {e}")
    return False


def check_stale_trade(trade: dict, pnl_pct: float) -> bool:
    """
    If position has been open > STALE_TRADE_HOURS and P&L is flat,
    it's a dead momentum play — exit to free capital.
    """
    entry_time = trade.get("entry_timestamp")
    if not entry_time:
        return False

    # Handle both timezone-aware and naive datetimes
    if entry_time.tzinfo is None:
        hours_held = (datetime.now() - entry_time).total_seconds() / 3600
    else:
        hours_held = (datetime.now(timezone.utc) - entry_time).total_seconds() / 3600

    if hours_held >= STALE_TRADE_HOURS:
        if STALE_TRADE_MIN_PNL <= pnl_pct <= STALE_TRADE_MAX_PNL:
            log.info(
                f"{trade.get('instrument') or trade.get('symbol')}: "
                f"stale trade — {hours_held:.1f}h held, pnl={pnl_pct * 100:+.2f}% "
                f"(flat zone {STALE_TRADE_MIN_PNL * 100}% to {STALE_TRADE_MAX_PNL * 100}%)"
            )
            return True
    return False


# ══════════════════════════════════════════════════════════════════════════════
# EMERGENCY EXIT
# ══════════════════════════════════════════════════════════════════════════════

def emergency_exit(ibkr: IBKRSellClient, symbol: str, qty: int,
                   reason: str) -> tuple[float, str]:
    """Aggressive limit → market fallback for urgent exits."""
    log.warning(f"{symbol}: EMERGENCY EXIT — {reason}")

    ticker      = ibkr.get_ticker(symbol)
    current_bid = ticker.bid or ticker.last or ticker.close

    if not current_bid:
        log.error(f"{symbol}: no price for emergency exit!")
        return 0.0, "exit_failed_no_price"

    # Try aggressive limit first
    trade    = ibkr.place_limit_sell(symbol, current_bid, qty, emergency=True)
    order_id = trade.order.orderId

    for _ in range(EMERGENCY_TIMEOUT):
        ibkr.ib.sleep(1)
        fill = ibkr.get_avg_fill_price_from_broker(symbol, order_id)
        if fill:
            log.info(f"{symbol}: emergency limit filled @ ${fill:.4f}")
            return fill, reason

    # Cancel and go market
    log.warning(f"{symbol}: limit not filled in {EMERGENCY_TIMEOUT}s → MARKET")
    try:
        ibkr.ib.cancelOrder(trade.order)
    except Exception:
        pass

    market_trade = ibkr.place_market_sell(symbol, qty)
    ibkr.ib.sleep(2)
    fill = ibkr.get_avg_fill_price_from_broker(symbol, market_trade.order.orderId)
    if fill:
        return fill, f"{reason}_market"

    return current_bid * 0.98, f"{reason}_estimated"


# ══════════════════════════════════════════════════════════════════════════════
# EXIT DECISION ENGINE
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class ExitDecision:
    """What the monitor loop should do for a given position."""
    action: str            # "hold", "partial_sell", "full_sell", "emergency_sell"
    reason: str            # human-readable reason
    qty_to_sell: int = 0   # how many shares to sell (0 = hold)


def evaluate_position(trade: dict, current_price: float, entry_price: float,
                      ibkr: IBKRSellClient) -> ExitDecision:
    """
    Master exit logic — evaluates all conditions in priority order.
    Returns an ExitDecision telling the monitor loop what to do.
    """
    symbol    = trade.get("instrument") or trade.get("symbol")
    total_qty = trade.get("quantity") or trade.get("qty")
    highest   = trade.get("highest_price", entry_price)
    phase     = trade.get("phase", "initial")  # "initial" or "runner"
    pnl_pct   = (current_price - entry_price) / entry_price

    # Track new high
    if current_price > highest:
        highest = current_price
        update_trade_field(trade["_id"], {"highest_price": highest})

    drawdown_from_high = (current_price - highest) / highest if highest > 0 else 0

    log.info(
        f"{symbol}: phase={phase} | entry=${entry_price:.4f} now=${current_price:.4f} "
        f"pnl={pnl_pct * 100:+.2f}% | high=${highest:.4f} dd={drawdown_from_high * 100:.2f}% "
        f"qty={total_qty}"
    )

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 1 — Hard stop (always active)
    # ══════════════════════════════════════════════════════════════════════
    if pnl_pct <= -HARD_STOP_PCT:
        return ExitDecision("emergency_sell", "hard_stop_10pct", total_qty)

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 2 — Flash crash (always active)
    # ══════════════════════════════════════════════════════════════════════
    flash = check_flash_crash(symbol, current_price, trade)
    if flash:
        return ExitDecision("emergency_sell", flash, total_qty)

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 3 — Liquidity crisis (always active)
    # ══════════════════════════════════════════════════════════════════════
    if check_liquidity_crisis(ibkr, symbol):
        return ExitDecision("emergency_sell", "liquidity_crisis", total_qty)

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 4 — Partial profit (only in "initial" phase)
    # ══════════════════════════════════════════════════════════════════════
    if phase == "initial" and pnl_pct >= PARTIAL_PROFIT_PCT:
        sell_qty = max(1, int(total_qty * PARTIAL_SELL_FRACTION))
        if sell_qty >= total_qty:
            # Position too small to split — just sell all
            return ExitDecision("full_sell", f"profit_target_{pnl_pct * 100:.1f}pct", total_qty)
        return ExitDecision("partial_sell", f"partial_profit_{pnl_pct * 100:.1f}pct", sell_qty)

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 5 — Trailing stop (primarily for runner phase, but also
    #              protects initial phase if price ran and pulled back)
    # ══════════════════════════════════════════════════════════════════════
    stop_level = compute_trailing_stop(entry_price, highest, pnl_pct)

    # Only activate trailing stop if we've been profitable enough
    # that the trail is meaningful (above hard stop level)
    if stop_level > entry_price * (1 - HARD_STOP_PCT):
        if current_price <= stop_level:
            trail_pct = get_trailing_stop_pct(pnl_pct)
            return ExitDecision(
                "full_sell",
                f"trailing_stop_{trail_pct * 100:.0f}pct_from_high",
                total_qty,
            )

    # Update trailing stop level in DB for monitoring
    update_trade_field(trade["_id"], {"trailing_stop": stop_level})

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 6 — Technical exit signals (runner phase)
    #   PSAR bearish + price below EMA 9 → exit
    #   EMA 9 death-crosses EMA 21        → exit
    # ══════════════════════════════════════════════════════════════════════
    if phase == "runner":
        signals = ibkr.compute_exit_signals(symbol)
        if signals:
            # PSAR bearish AND price below EMA 9 = trend broken
            if signals.get("psar_bearish") and signals.get("price_below_ema9"):
                return ExitDecision(
                    "full_sell",
                    "psar_bearish_below_ema9",
                    total_qty,
                )

            # EMA 9/21 death cross = momentum dead
            if signals.get("ema_death_cross"):
                return ExitDecision(
                    "full_sell",
                    "ema_9_21_death_cross",
                    total_qty,
                )

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 7 — Stale trade (going nowhere for too long)
    # ══════════════════════════════════════════════════════════════════════
    if check_stale_trade(trade, pnl_pct):
        return ExitDecision("full_sell", "stale_trade_timeout", total_qty)

    # ══════════════════════════════════════════════════════════════════════
    # DEFAULT — Hold
    # ══════════════════════════════════════════════════════════════════════
    trail_pct = get_trailing_stop_pct(pnl_pct)
    log.info(
        f"{symbol}: HOLD | trail={trail_pct * 100:.0f}% "
        f"stop=${stop_level:.4f} | phase={phase}"
    )
    return ExitDecision("hold", "holding", 0)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN MONITOR LOOP
# ══════════════════════════════════════════════════════════════════════════════

def monitor_positions(ibkr: IBKRSellClient):
    log.info("=" * 60)
    log.info("SELL BOT v2 — Smart Position Manager")
    log.info(f"Hard Stop: {HARD_STOP_PCT * 100}% | "
             f"Partial: {PARTIAL_PROFIT_PCT * 100}% ({PARTIAL_SELL_FRACTION * 100}% of qty) | "
             f"Flash: {FLASH_CRASH_PCT * 100}%")
    log.info(f"Trailing tiers: {[(f'+{t*100:.0f}%', f'{p*100:.0f}%') for t, p in TRAILING_TIERS]}")
    log.info(f"Stale exit: >{STALE_TRADE_HOURS}h if PnL in "
             f"[{STALE_TRADE_MIN_PNL * 100}%, {STALE_TRADE_MAX_PNL * 100}%]")
    log.info("=" * 60)

    while True:
        try:
            open_trades = get_open_trades()
            log.info(f"--- Checking {len(open_trades)} open trade(s) ---")

            for trade in open_trades:
                symbol    = trade.get("instrument") or trade.get("symbol")
                total_qty = trade.get("quantity") or trade.get("qty")

                if not symbol or not total_qty:
                    log.warning(f"Skipping trade {trade['_id']}: missing symbol or qty")
                    continue

                # Get entry and current price
                entry_price   = get_effective_entry_price(trade, ibkr)
                current_price = ibkr.get_last_price(symbol)
                if not current_price:
                    log.warning(f"{symbol}: no current price, skipping")
                    continue

                # Evaluate what to do
                decision = evaluate_position(trade, current_price, entry_price, ibkr)

                if decision.action == "hold":
                    continue

                # ── Execute the decision ──────────────────────────────────────

                if decision.action == "emergency_sell":
                    exit_price, final_reason = emergency_exit(
                        ibkr, symbol, decision.qty_to_sell, decision.reason
                    )
                    close_trade(trade, exit_price, final_reason, decision.qty_to_sell)

                elif decision.action == "partial_sell":
                    sell_trade   = ibkr.place_limit_sell(symbol, current_price,
                                                         decision.qty_to_sell)
                    sell_order_id = sell_trade.order.orderId
                    confirmed     = ibkr.get_sell_fill_price(symbol, sell_order_id,
                                                              current_price)
                    close_trade(trade, confirmed, decision.reason, decision.qty_to_sell)

                elif decision.action == "full_sell":
                    sell_trade   = ibkr.place_limit_sell(symbol, current_price,
                                                         decision.qty_to_sell)
                    sell_order_id = sell_trade.order.orderId
                    confirmed     = ibkr.get_sell_fill_price(symbol, sell_order_id,
                                                              current_price)
                    close_trade(trade, confirmed, decision.reason, decision.qty_to_sell)

        except Exception as e:
            log.error(f"Monitor error: {e}", exc_info=True)

        time.sleep(MONITOR_INTERVAL)


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

def main():
    log.info("=== Sell Bot v2 Starting ===")
    ibkr = IBKRSellClient()

    try:
        ibkr.connect()
        monitor_positions(ibkr)
    except KeyboardInterrupt:
        log.info("Sell bot stopped by user.")
    except Exception as e:
        log.error(f"Fatal error: {e}", exc_info=True)
    finally:
        ibkr.disconnect()
        log.info("IBKR disconnected.")


if __name__ == "__main__":
    main()


In [ ]:
"""
Sell Bot v2 — Smart Position Manager
═════════════════════════════════════
Monitors open positions via IBKR + MongoDB.

EXIT STRATEGY (designed for momentum breakout plays):
─────────────────────────────────────────────────────
Phase 1 — PROTECTION (from entry until profitable):
  • Hard stop: -10% from entry (expanded from 3% to let breakouts breathe)
  • Flash crash: -5% drop in a single 30-second interval → emergency exit

Phase 2 — PARTIAL PROFIT at +50%:
  • Sell 50% of position, lock in gains
  • Remaining 50% enters "runner" mode with trailing stop

Phase 3 — RUNNER (remaining shares after partial):
  • Trailing stop tightens in tiers based on how far price has run:
      +50% to +75%  →  trail 15% from high  (give it room to consolidate)
      +75% to +100% →  trail 12% from high
      +100% to +150% → trail 10% from high
      +150%+         → trail  8% from high  (tightest — protect big win)
  • TRAILING STOP CONFIRMATION: trail stop only triggers if PSAR is bearish
    OR price is below EMA 21 (prevents exiting on normal pullbacks in a trend)
  • PSAR flip bearish → soft exit (only triggers if also below EMA 9)
  • EMA 9 crosses below EMA 21 → exit runner (trend is done)

Phase 4 — TIME DECAY (if stock goes nowhere):
  • If position is between -5% and +10% after 2 hours → exit
    (momentum play didn't work, free up capital)

EMERGENCY EXITS (always active, any phase):
  • Flash crash detection (configurable threshold)
  • Liquidity crisis (spread > 0.5%)
  • These use aggressive limit → market order fallback

RE-ENTRY COOLDOWN:
  • After closing a position, a cooldown period is recorded in MongoDB
  • Buy bot should check cooldown before re-entering the same symbol
"""

import logging
import sys
import time
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta
import random
from typing import Optional
from dataclasses import dataclass

from pymongo import MongoClient
from ib_insync import *

import warnings
warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# ─── IBKR ─────────────────────────────────────────────────────────────────────
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)

# ─── Exit Thresholds ──────────────────────────────────────────────────────────
HARD_STOP_PCT          = 0.10    # 10% max loss from entry (expanded from 3%)
FLASH_CRASH_PCT        = 0.05    # 5% drop in one interval → emergency
LIQUIDITY_SPREAD_PCT   = 0.005   # 0.5% spread → liquidity crisis

# ─── Partial Profit ───────────────────────────────────────────────────────────
PARTIAL_PROFIT_PCT     = 0.50    # Take partial at +50%
PARTIAL_SELL_FRACTION  = 0.50    # Sell 50% of position

# ─── Tiered Trailing Stops (for runner after partial) ─────────────────────────
# Each tuple: (profit_threshold, trail_pct_from_high)
# Evaluated top-down — first match wins
TRAILING_TIERS = [
    (1.50, 0.08),   # +150%+ profit → trail 8% from high
    (1.00, 0.10),   # +100%+ profit → trail 10%
    (0.75, 0.12),   # +75%+  profit → trail 12%
    (0.50, 0.15),   # +50%+  profit → trail 15%
]
DEFAULT_TRAILING_PCT   = 0.20    # Before +50%, if somehow needed: 20%

# ─── Time Decay ───────────────────────────────────────────────────────────────
STALE_TRADE_HOURS      = 2      # Exit if position stagnates after this long
STALE_TRADE_MIN_PNL    = -0.05  # … and P&L is between this …
STALE_TRADE_MAX_PNL    = 0.10   # … and this (going nowhere)

# ─── Technical Exit Signals ───────────────────────────────────────────────────
# PSAR + EMA crossover used as confirmation for runner exits
EMA_FAST_PERIOD        = 9
EMA_SLOW_PERIOD        = 21

# ─── Re-entry Cooldown ───────────────────────────────────────────────────────
REENTRY_COOLDOWN_MIN   = 30     # Minutes to wait before re-entering same symbol

# ─── Monitoring ───────────────────────────────────────────────────────────────
MONITOR_INTERVAL       = 30     # seconds between checks
EMERGENCY_TIMEOUT      = 5      # seconds to wait for limit fill before market

# ─── MongoDB ──────────────────────────────────────────────────────────────────
mongo_client       = MongoClient("mongodb://localhost:27017/")
db                 = mongo_client["breakout_db"]
trades_collection  = db["trades_v2"]
cooldown_collection = db["cooldowns"]   # NEW: tracks re-entry cooldowns


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING
# ══════════════════════════════════════════════════════════════════════════════

class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh  = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh  = logging.FileHandler("sell_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("sell_bot")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


# ══════════════════════════════════════════════════════════════════════════════
# IBKR CLIENT
# ══════════════════════════════════════════════════════════════════════════════

class IBKRSellClient:
    def __init__(self):
        util.startLoop()
        self.ib = IB()

    def connect(self):
        self.ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
        log.info(f"IBKR connected | accounts: {self.ib.wrapper.accounts}")

    def disconnect(self):
        self.ib.disconnect()

    # ── Price helpers ─────────────────────────────────────────────────────────

    def get_last_price(self, symbol: str) -> Optional[float]:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "106", False, False)
        self.ib.sleep(3)
        self.ib.cancelMktData(contract)

        for label, val in [("last", ticker.last), ("ask", ticker.ask),
                           ("bid", ticker.bid), ("close", ticker.close)]:
            if val and float(val) > 0:
                log.debug(f"{symbol}: price source='{label}' ${float(val):.4f}")
                return float(val)

        log.warning(f"{symbol}: no price available")
        return None

    def get_ticker(self, symbol: str) -> Ticker:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "", False, False)
        self.ib.sleep(2)
        return ticker

    # ── Historical bars ───────────────────────────────────────────────────────

    def get_bars(self, symbol: str, n: int = 70) -> Optional[pd.DataFrame]:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)

        for what_to_show in ("TRADES", "MIDPOINT"):
            raw = self.ib.reqHistoricalData(
                contract,
                endDateTime="",
                durationStr="10 D",
                barSizeSetting="5 mins",
                whatToShow=what_to_show,
                useRTH=False,
                keepUpToDate=False,
            )
            if raw:
                df = pd.DataFrame([
                    {"open": b.open, "high": b.high, "low": b.low,
                     "close": b.close, "volume": getattr(b, "volume", 0)}
                    for b in raw
                ])
                return df.tail(n).reset_index(drop=True)

        log.warning(f"{symbol}: no bars returned")
        return None

    # ── Technical indicators ──────────────────────────────────────────────────

    def compute_exit_signals(self, symbol: str) -> dict:
        """
        Compute PSAR, EMA 9, EMA 21 for exit decisions.
        Returns dict with signal flags or empty dict on failure.
        """
        df = self.get_bars(symbol, n=70)
        if df is None or len(df) < 25:
            log.warning(f"{symbol}: not enough bars for exit signals")
            return {}

        # Parabolic SAR
        psar_vals = self._compute_psar(df["high"].values, df["low"].values,
                                       df["close"].values)

        # EMA 9 / 21
        ema_fast = df["close"].ewm(span=EMA_FAST_PERIOD, adjust=False).mean()
        ema_slow = df["close"].ewm(span=EMA_SLOW_PERIOD, adjust=False).mean()

        price     = float(df["close"].iloc[-1])
        psar_now  = float(psar_vals[-1])
        ema9_now  = float(ema_fast.iloc[-1])
        ema21_now = float(ema_slow.iloc[-1])

        # Previous bar for crossover detection
        ema9_prev  = float(ema_fast.iloc[-2])
        ema21_prev = float(ema_slow.iloc[-2])

        psar_bearish       = psar_now > price       # SAR above price = downtrend
        ema_death_cross    = (ema9_prev >= ema21_prev) and (ema9_now < ema21_now)
        ema_bearish        = ema9_now < ema21_now    # EMA 9 below 21
        price_below_ema9   = price < ema9_now
        price_below_ema21  = price < ema21_now       # NEW: for trailing stop confirmation

        signals = {
            "price":              price,
            "psar":               psar_now,
            "ema_9":              ema9_now,
            "ema_21":             ema21_now,
            "psar_bearish":       psar_bearish,
            "ema_death_cross":    ema_death_cross,
            "ema_bearish":        ema_bearish,
            "price_below_ema9":   price_below_ema9,
            "price_below_ema21":  price_below_ema21,
        }

        log.info(
            f"{symbol} signals: PSAR={psar_now:.4f}({'BEAR' if psar_bearish else 'BULL'}) "
            f"EMA9={ema9_now:.4f} EMA21={ema21_now:.4f} "
            f"cross={'YES' if ema_death_cross else 'no'} "
            f"below9={'YES' if price_below_ema9 else 'no'} "
            f"below21={'YES' if price_below_ema21 else 'no'}"
        )
        return signals

    @staticmethod
    def _compute_psar(highs, lows, closes, iaf=0.02, max_af=0.2) -> np.ndarray:
        n    = len(closes)
        psar = closes.copy().astype(float)
        bull = True
        af   = iaf
        hp   = float(highs[0])
        lp   = float(lows[0])

        for i in range(2, n):
            if bull:
                psar[i] = psar[i - 1] + af * (hp - psar[i - 1])
                psar[i] = min(psar[i], lows[i - 1], lows[i - 2])
                if lows[i] < psar[i]:
                    bull    = False
                    psar[i] = hp
                    lp      = lows[i]
                    af      = iaf
                else:
                    if highs[i] > hp:
                        hp = highs[i]
                        af = min(af + iaf, max_af)
            else:
                psar[i] = psar[i - 1] + af * (lp - psar[i - 1])
                psar[i] = max(psar[i], highs[i - 1], highs[i - 2])
                if highs[i] > psar[i]:
                    bull    = True
                    psar[i] = lp
                    hp      = highs[i]
                    af      = iaf
                else:
                    if lows[i] < lp:
                        lp = lows[i]
                        af = min(af + iaf, max_af)
        return psar

    # ── Order execution ───────────────────────────────────────────────────────

    def get_avg_fill_price_from_broker(self, symbol: str, order_id: int) -> Optional[float]:
        for trade in self.ib.trades():
            if (trade.contract.symbol == symbol
                    and trade.order.orderId == order_id
                    and trade.orderStatus.avgFillPrice > 0):
                return float(trade.orderStatus.avgFillPrice)

        fills   = self.ib.fills()
        matched = [f for f in fills
                   if f.contract.symbol == symbol and f.execution.orderId == order_id]
        if matched:
            total_qty  = sum(f.execution.shares for f in matched)
            total_cost = sum(f.execution.shares * f.execution.price for f in matched)
            return total_cost / total_qty

        return None

    def place_limit_sell(self, symbol: str, price: float, qty: int,
                         emergency: bool = False) -> Trade:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)

        if emergency:
            price = price * 0.99  # 1% below for faster fill

        order = LimitOrder("SELL", qty, round(price, 2))
        order.outsideRth = True
        trade = self.ib.placeOrder(contract, order)
        self.ib.sleep(1)

        log.info(f"SELL LIMIT {symbol} x{qty} @ ${price:.2f} "
                 f"orderId={trade.order.orderId} emergency={emergency}")
        return trade

    def place_market_sell(self, symbol: str, qty: int) -> Trade:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        order = MarketOrder("SELL", qty)
        trade = self.ib.placeOrder(contract, order)
        self.ib.sleep(1)
        log.info(f"SELL MARKET {symbol} x{qty} orderId={trade.order.orderId}")
        return trade

    def get_sell_fill_price(self, symbol: str, sell_order_id: int,
                            fallback_price: float) -> float:
        self.ib.sleep(3)
        avg = self.get_avg_fill_price_from_broker(symbol, sell_order_id)
        if avg:
            return avg
        log.warning(f"{symbol}: sell fill not confirmed, using fallback ${fallback_price:.4f}")
        return fallback_price


# ══════════════════════════════════════════════════════════════════════════════
# MONGODB HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def get_open_trades() -> list:
    return list(trades_collection.find({"status": "open"}))


def get_effective_entry_price(trade: dict, ibkr: IBKRSellClient) -> float:
    """Priority: avg_fill_price (DB) → broker lookup → entry_price (DB)."""
    if trade.get("avg_fill_price"):
        return float(trade["avg_fill_price"])

    order_id = trade.get("order_id")
    symbol   = trade.get("instrument") or trade.get("symbol")

    if order_id:
        broker_avg = ibkr.get_avg_fill_price_from_broker(symbol, order_id)
        if broker_avg:
            trades_collection.update_one(
                {"_id": trade["_id"]},
                {"$set": {"avg_fill_price": broker_avg, "entry_price": broker_avg}},
            )
            return broker_avg

    return float(trade["entry_price"])


def update_trade_field(trade_id, updates: dict):
    updates["updated_at"] = datetime.now(timezone.utc)
    trades_collection.update_one({"_id": trade_id}, {"$set": updates})


def close_trade(trade: dict, exit_price: float, reason: str, qty_sold: int = None):
    """Close (or partially close) a trade in MongoDB."""
    symbol      = trade.get("instrument") or trade.get("symbol")
    entry_price = float(trade.get("avg_fill_price") or trade["entry_price"])
    total_qty   = trade.get("quantity") or trade.get("qty")
    pnl_pct     = (exit_price - entry_price) / entry_price

    if qty_sold is None:
        qty_sold = total_qty

    realized_pnl = (exit_price - entry_price) * qty_sold

    if qty_sold >= total_qty:
        # Full close
        trades_collection.update_one(
            {"_id": trade["_id"]},
            {"$set": {
                "status":           "closed",
                "exit_price":       exit_price,
                "exit_timestamp":   datetime.now(timezone.utc),
                "reason":           reason,
                "pnl_pct":          round(pnl_pct * 100, 2),
                "realized_pnl":     realized_pnl,
                "updated_at":       datetime.now(timezone.utc),
            }},
        )
        log.info(
            f"CLOSED {symbol} | entry=${entry_price:.4f} exit=${exit_price:.4f} "
            f"pnl={pnl_pct * 100:+.2f}% | ${realized_pnl:+.2f} | reason={reason}"
        )
        # ── Set re-entry cooldown ─────────────────────────────────────────
        set_cooldown(symbol, reason)
    else:
        # Partial close — reduce quantity, record partial exit
        remaining = total_qty - qty_sold
        trades_collection.update_one(
            {"_id": trade["_id"]},
            {"$set": {
                "quantity":              remaining,
                "partial_exit_price":    exit_price,
                "partial_exit_qty":      qty_sold,
                "partial_exit_time":     datetime.now(timezone.utc),
                "partial_exit_reason":   reason,
                "partial_realized_pnl":  realized_pnl,
                "phase":                 "runner",
                "updated_at":            datetime.now(timezone.utc),
            }},
        )
        log.info(
            f"PARTIAL CLOSE {symbol} | sold {qty_sold}/{total_qty} @ ${exit_price:.4f} "
            f"pnl={pnl_pct * 100:+.2f}% | ${realized_pnl:+.2f} | "
            f"remaining={remaining} shares → runner mode"
        )


# ══════════════════════════════════════════════════════════════════════════════
# RE-ENTRY COOLDOWN
# ══════════════════════════════════════════════════════════════════════════════

def set_cooldown(symbol: str, reason: str):
    """
    After closing a position, set a cooldown to prevent immediate re-entry.
    The buy bot should call is_in_cooldown() before opening a new position.
    """
    expires_at = datetime.now(timezone.utc) + timedelta(minutes=REENTRY_COOLDOWN_MIN)
    cooldown_collection.update_one(
        {"symbol": symbol},
        {"$set": {
            "symbol":       symbol,
            "reason":       reason,
            "closed_at":    datetime.now(timezone.utc),
            "expires_at":   expires_at,
            "cooldown_min": REENTRY_COOLDOWN_MIN,
        }},
        upsert=True,
    )
    log.info(
        f"{symbol}: COOLDOWN SET — no re-entry for {REENTRY_COOLDOWN_MIN} min "
        f"(until {expires_at.strftime('%H:%M:%S')} UTC) | close reason: {reason}"
    )


def is_in_cooldown(symbol: str) -> bool:
    """
    Check if a symbol is in cooldown (call this from the BUY bot before entering).
    Returns True if the symbol should NOT be traded yet.
    """
    record = cooldown_collection.find_one({"symbol": symbol})
    if not record:
        return False

    expires_at = record.get("expires_at")
    if not expires_at:
        return False

    now = datetime.now(timezone.utc)
    if expires_at.tzinfo is None:
        # Handle naive datetime from DB
        in_cooldown = datetime.now() < expires_at
    else:
        in_cooldown = now < expires_at

    if in_cooldown:
        remaining = (expires_at - now).total_seconds() / 60
        log.info(f"{symbol}: IN COOLDOWN — {remaining:.1f} min remaining "
                 f"(closed for: {record.get('reason', 'unknown')})")
    else:
        # Cooldown expired — clean up
        cooldown_collection.delete_one({"symbol": symbol})

    return in_cooldown


def clear_cooldown(symbol: str):
    """Manually clear a cooldown (e.g., if conditions change dramatically)."""
    cooldown_collection.delete_one({"symbol": symbol})
    log.info(f"{symbol}: cooldown manually cleared")


# ══════════════════════════════════════════════════════════════════════════════
# RISK MANAGEMENT
# ══════════════════════════════════════════════════════════════════════════════

def get_trailing_stop_pct(pnl_pct: float) -> float:
    """
    Return the trailing stop percentage based on how far price has run.
    Tighter trail as profit grows (protect bigger wins).
    """
    for threshold, trail in TRAILING_TIERS:
        if pnl_pct >= threshold:
            return trail
    return DEFAULT_TRAILING_PCT


def compute_trailing_stop(entry_price: float, highest_price: float,
                          current_pnl_pct: float) -> float:
    """Compute the effective trailing stop price."""
    trail_pct  = get_trailing_stop_pct(current_pnl_pct)
    trail_stop = highest_price * (1 - trail_pct)
    hard_stop  = entry_price * (1 - HARD_STOP_PCT)
    return max(trail_stop, hard_stop)


def check_flash_crash(symbol: str, current_price: float, trade: dict) -> Optional[str]:
    """Detect sudden price drops within a single monitoring interval."""
    last_price = trade.get("last_price")

    # Always update for next iteration
    update_trade_field(trade["_id"], {
        "last_price":      current_price,
        "last_price_time": datetime.now(timezone.utc),
    })

    if last_price and last_price > 0:
        drop_pct = (current_price - last_price) / last_price
        if drop_pct <= -FLASH_CRASH_PCT:
            log.warning(f"{symbol}: FLASH CRASH {drop_pct * 100:.2f}% in {MONITOR_INTERVAL}s")
            return f"flash_crash_{abs(drop_pct) * 100:.1f}pct"

    return None


def check_liquidity_crisis(ibkr: IBKRSellClient, symbol: str) -> bool:
    """Detect bid-ask spread blowing out."""
    try:
        ticker = ibkr.get_ticker(symbol)
        if ticker.bid and ticker.ask and ticker.bid > 0:
            mid        = (ticker.ask + ticker.bid) / 2
            spread_pct = (ticker.ask - ticker.bid) / mid
            if spread_pct > LIQUIDITY_SPREAD_PCT:
                log.warning(f"{symbol}: liquidity crisis — spread {spread_pct * 100:.2f}%")
                return True
    except Exception as e:
        log.error(f"{symbol}: liquidity check error — {e}")
    return False


def check_stale_trade(trade: dict, pnl_pct: float) -> bool:
    """
    If position has been open > STALE_TRADE_HOURS and P&L is flat,
    it's a dead momentum play — exit to free capital.
    """
    entry_time = trade.get("entry_timestamp")
    if not entry_time:
        return False

    # Handle both timezone-aware and naive datetimes
    if entry_time.tzinfo is None:
        hours_held = (datetime.now() - entry_time).total_seconds() / 3600
    else:
        hours_held = (datetime.now(timezone.utc) - entry_time).total_seconds() / 3600

    if hours_held >= STALE_TRADE_HOURS:
        if STALE_TRADE_MIN_PNL <= pnl_pct <= STALE_TRADE_MAX_PNL:
            log.info(
                f"{trade.get('instrument') or trade.get('symbol')}: "
                f"stale trade — {hours_held:.1f}h held, pnl={pnl_pct * 100:+.2f}% "
                f"(flat zone {STALE_TRADE_MIN_PNL * 100}% to {STALE_TRADE_MAX_PNL * 100}%)"
            )
            return True
    return False


# ══════════════════════════════════════════════════════════════════════════════
# EMERGENCY EXIT
# ══════════════════════════════════════════════════════════════════════════════

def emergency_exit(ibkr: IBKRSellClient, symbol: str, qty: int,
                   reason: str) -> tuple[float, str]:
    """Aggressive limit → market fallback for urgent exits."""
    log.warning(f"{symbol}: EMERGENCY EXIT — {reason}")

    ticker      = ibkr.get_ticker(symbol)
    current_bid = ticker.bid or ticker.last or ticker.close

    if not current_bid:
        log.error(f"{symbol}: no price for emergency exit!")
        return 0.0, "exit_failed_no_price"

    # Try aggressive limit first
    trade    = ibkr.place_limit_sell(symbol, current_bid, qty, emergency=True)
    order_id = trade.order.orderId

    for _ in range(EMERGENCY_TIMEOUT):
        ibkr.ib.sleep(1)
        fill = ibkr.get_avg_fill_price_from_broker(symbol, order_id)
        if fill:
            log.info(f"{symbol}: emergency limit filled @ ${fill:.4f}")
            return fill, reason

    # Cancel and go market
    log.warning(f"{symbol}: limit not filled in {EMERGENCY_TIMEOUT}s → MARKET")
    try:
        ibkr.ib.cancelOrder(trade.order)
    except Exception:
        pass

    market_trade = ibkr.place_market_sell(symbol, qty)
    ibkr.ib.sleep(2)
    fill = ibkr.get_avg_fill_price_from_broker(symbol, market_trade.order.orderId)
    if fill:
        return fill, f"{reason}_market"

    return current_bid * 0.98, f"{reason}_estimated"


# ══════════════════════════════════════════════════════════════════════════════
# EXIT DECISION ENGINE
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class ExitDecision:
    """What the monitor loop should do for a given position."""
    action: str            # "hold", "partial_sell", "full_sell", "emergency_sell"
    reason: str            # human-readable reason
    qty_to_sell: int = 0   # how many shares to sell (0 = hold)


def evaluate_position(trade: dict, current_price: float, entry_price: float,
                      ibkr: IBKRSellClient) -> ExitDecision:
    """
    Master exit logic — evaluates all conditions in priority order.
    Returns an ExitDecision telling the monitor loop what to do.
    """
    symbol    = trade.get("instrument") or trade.get("symbol")
    total_qty = trade.get("quantity") or trade.get("qty")
    highest   = trade.get("highest_price", entry_price)
    phase     = trade.get("phase", "initial")  # "initial" or "runner"
    pnl_pct   = (current_price - entry_price) / entry_price

    # Track new high
    if current_price > highest:
        highest = current_price
        update_trade_field(trade["_id"], {"highest_price": highest})

    drawdown_from_high = (current_price - highest) / highest if highest > 0 else 0

    log.info(
        f"{symbol}: phase={phase} | entry=${entry_price:.4f} now=${current_price:.4f} "
        f"pnl={pnl_pct * 100:+.2f}% | high=${highest:.4f} dd={drawdown_from_high * 100:.2f}% "
        f"qty={total_qty}"
    )

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 1 — Hard stop (always active)
    # ══════════════════════════════════════════════════════════════════════
    if pnl_pct <= -HARD_STOP_PCT:
        return ExitDecision("emergency_sell", "hard_stop_10pct", total_qty)

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 2 — Flash crash (always active)
    # ══════════════════════════════════════════════════════════════════════
    flash = check_flash_crash(symbol, current_price, trade)
    if flash:
        return ExitDecision("emergency_sell", flash, total_qty)

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 3 — Liquidity crisis (always active)
    # ══════════════════════════════════════════════════════════════════════
    if check_liquidity_crisis(ibkr, symbol):
        return ExitDecision("emergency_sell", "liquidity_crisis", total_qty)

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 4 — Partial profit (only in "initial" phase)
    # ══════════════════════════════════════════════════════════════════════
    if phase == "initial" and pnl_pct >= PARTIAL_PROFIT_PCT:
        sell_qty = max(1, int(total_qty * PARTIAL_SELL_FRACTION))
        if sell_qty >= total_qty:
            # Position too small to split — just sell all
            return ExitDecision("full_sell", f"profit_target_{pnl_pct * 100:.1f}pct", total_qty)
        return ExitDecision("partial_sell", f"partial_profit_{pnl_pct * 100:.1f}pct", sell_qty)

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 5 — Trailing stop WITH CONFIRMATION
    #   Trail stop level is still computed, but it only triggers if
    #   PSAR is bearish OR price is below EMA 21.
    #   This prevents exiting on normal pullbacks during a strong trend.
    # ══════════════════════════════════════════════════════════════════════
    stop_level = compute_trailing_stop(entry_price, highest, pnl_pct)

    # Only activate trailing stop if we've been profitable enough
    # that the trail is meaningful (above hard stop level)
    if stop_level > entry_price * (1 - HARD_STOP_PCT):
        if current_price <= stop_level:
            # ── NEW: Require technical confirmation before triggering ──
            signals = ibkr.compute_exit_signals(symbol)
            trend_broken = False

            if signals:
                psar_bearish      = signals.get("psar_bearish", False)
                price_below_ema21 = signals.get("price_below_ema21", False)

                if psar_bearish or price_below_ema21:
                    trend_broken = True
                    confirm_reason = (
                        "psar_bearish" if psar_bearish else "below_ema21"
                    )
                    log.info(
                        f"{symbol}: trailing stop CONFIRMED by {confirm_reason} "
                        f"(PSAR_bear={psar_bearish}, below_EMA21={price_below_ema21})"
                    )
                else:
                    log.info(
                        f"{symbol}: trailing stop level breached (${stop_level:.4f}) "
                        f"but PSAR bullish & above EMA21 — HOLDING through pullback"
                    )
            else:
                # No signals available (bars failed) — fall back to triggering
                # the trailing stop without confirmation for safety
                trend_broken = True
                log.warning(
                    f"{symbol}: trailing stop breached & no signal data — "
                    f"triggering stop as safety fallback"
                )

            if trend_broken:
                trail_pct = get_trailing_stop_pct(pnl_pct)
                return ExitDecision(
                    "full_sell",
                    f"trailing_stop_{trail_pct * 100:.0f}pct_confirmed",
                    total_qty,
                )

    # Update trailing stop level in DB for monitoring
    update_trade_field(trade["_id"], {"trailing_stop": stop_level})

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 6 — Technical exit signals (runner phase)
    #   PSAR bearish + price below EMA 9 → exit
    #   EMA 9 death-crosses EMA 21        → exit
    # ══════════════════════════════════════════════════════════════════════
    if phase == "runner":
        signals = ibkr.compute_exit_signals(symbol)
        if signals:
            # PSAR bearish AND price below EMA 9 = trend broken
            if signals.get("psar_bearish") and signals.get("price_below_ema9"):
                return ExitDecision(
                    "full_sell",
                    "psar_bearish_below_ema9",
                    total_qty,
                )

            # EMA 9/21 death cross = momentum dead
            if signals.get("ema_death_cross"):
                return ExitDecision(
                    "full_sell",
                    "ema_9_21_death_cross",
                    total_qty,
                )

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 7 — Stale trade (going nowhere for too long)
    # ══════════════════════════════════════════════════════════════════════
    if check_stale_trade(trade, pnl_pct):
        return ExitDecision("full_sell", "stale_trade_timeout", total_qty)

    # ══════════════════════════════════════════════════════════════════════
    # DEFAULT — Hold
    # ══════════════════════════════════════════════════════════════════════
    trail_pct = get_trailing_stop_pct(pnl_pct)
    log.info(
        f"{symbol}: HOLD | trail={trail_pct * 100:.0f}% "
        f"stop=${stop_level:.4f} | phase={phase}"
    )
    return ExitDecision("hold", "holding", 0)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN MONITOR LOOP
# ══════════════════════════════════════════════════════════════════════════════

def monitor_positions(ibkr: IBKRSellClient):
    log.info("=" * 60)
    log.info("SELL BOT v2.1 — Smart Position Manager (PSAR-confirmed trails)")
    log.info(f"Hard Stop: {HARD_STOP_PCT * 100}% | "
             f"Partial: {PARTIAL_PROFIT_PCT * 100}% ({PARTIAL_SELL_FRACTION * 100}% of qty) | "
             f"Flash: {FLASH_CRASH_PCT * 100}%")
    log.info(f"Trailing tiers: {[(f'+{t*100:.0f}%', f'{p*100:.0f}%') for t, p in TRAILING_TIERS]}")
    log.info(f"Trail confirmation: PSAR bearish OR price < EMA 21")
    log.info(f"Re-entry cooldown: {REENTRY_COOLDOWN_MIN} min")
    log.info(f"Stale exit: >{STALE_TRADE_HOURS}h if PnL in "
             f"[{STALE_TRADE_MIN_PNL * 100}%, {STALE_TRADE_MAX_PNL * 100}%]")
    log.info("=" * 60)

    while True:
        try:
            open_trades = get_open_trades()
            log.info(f"--- Checking {len(open_trades)} open trade(s) ---")

            for trade in open_trades:
                symbol    = trade.get("instrument") or trade.get("symbol")
                total_qty = trade.get("quantity") or trade.get("qty")

                if not symbol or not total_qty:
                    log.warning(f"Skipping trade {trade['_id']}: missing symbol or qty")
                    continue

                # Get entry and current price
                entry_price   = get_effective_entry_price(trade, ibkr)
                current_price = ibkr.get_last_price(symbol)
                if not current_price:
                    log.warning(f"{symbol}: no current price, skipping")
                    continue

                # Evaluate what to do
                decision = evaluate_position(trade, current_price, entry_price, ibkr)

                if decision.action == "hold":
                    continue

                # ── Execute the decision ──────────────────────────────────────

                if decision.action == "emergency_sell":
                    exit_price, final_reason = emergency_exit(
                        ibkr, symbol, decision.qty_to_sell, decision.reason
                    )
                    close_trade(trade, exit_price, final_reason, decision.qty_to_sell)

                elif decision.action == "partial_sell":
                    sell_trade   = ibkr.place_limit_sell(symbol, current_price,
                                                         decision.qty_to_sell)
                    sell_order_id = sell_trade.order.orderId
                    confirmed     = ibkr.get_sell_fill_price(symbol, sell_order_id,
                                                              current_price)
                    close_trade(trade, confirmed, decision.reason, decision.qty_to_sell)

                elif decision.action == "full_sell":
                    sell_trade   = ibkr.place_limit_sell(symbol, current_price,
                                                         decision.qty_to_sell)
                    sell_order_id = sell_trade.order.orderId
                    confirmed     = ibkr.get_sell_fill_price(symbol, sell_order_id,
                                                              current_price)
                    close_trade(trade, confirmed, decision.reason, decision.qty_to_sell)

        except Exception as e:
            log.error(f"Monitor error: {e}", exc_info=True)

        time.sleep(MONITOR_INTERVAL)


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

def main():
    log.info("=== Sell Bot v2.1 Starting ===")
    ibkr = IBKRSellClient()

    try:
        ibkr.connect()
        monitor_positions(ibkr)
    except KeyboardInterrupt:
        log.info("Sell bot stopped by user.")
    except Exception as e:
        log.error(f"Fatal error: {e}", exc_info=True)
    finally:
        ibkr.disconnect()
        log.info("IBKR disconnected.")


if __name__ == "__main__":
    main()


In [ ]:
"""
Sell Bot v2.3 — Smart Position Manager
═══════════════════════════════════════
Monitors open positions via IBKR + MongoDB.

CHANGES from v2.1:
─────────────────
1. REMOVED liquidity crisis check entirely (was causing false exits on
   low-priced stocks where wide spreads are normal).
2. Profitable exits now rely on PSAR confirmation — trailing stop only
   triggers when PSAR is bearish (not EMA21 alone).
3. Duplicate position consolidation added — merges multiple open trades
   for the same symbol into one so exit logic works on full position.

EXIT STRATEGY (designed for momentum breakout plays):
─────────────────────────────────────────────────────
Phase 1 — PROTECTION (from entry until profitable):
  • Hard stop: -10% from entry
  • Flash crash: -5% drop in a single 30-second interval → emergency exit

Phase 2 — PARTIAL PROFIT at +50%:
  • Sell 50% of position, lock in gains
  • Remaining 50% enters "runner" mode with trailing stop

Phase 3 — RUNNER (remaining shares after partial):
  • Trailing stop tightens in tiers based on how far price has run:
      +50% to +75%  →  trail 15% from high
      +75% to +100% →  trail 12% from high
      +100% to +150% → trail 10% from high
      +150%+         → trail  8% from high
  • TRAILING STOP CONFIRMATION: trail stop only triggers if PSAR is bearish
    (prevents exiting on normal pullbacks in a trend)
  • PSAR flip bearish + price below EMA 9 → exit runner
  • EMA 9 crosses below EMA 21 → exit runner (trend is done)

Phase 4 — TIME DECAY (if stock goes nowhere):
  • If position is between -5% and +10% after 2 hours → exit

EMERGENCY EXITS (always active, any phase):
  • Hard stop (-10%)
  • Flash crash detection

RE-ENTRY COOLDOWN:
  • After closing a position, a cooldown period is recorded in MongoDB
  • Buy bot should check cooldown before re-entering the same symbol
"""

import logging
import sys
import time
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta
import random
from typing import Optional
from dataclasses import dataclass
from collections import defaultdict

from pymongo import MongoClient
from ib_insync import *

import warnings
warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# ─── IBKR ─────────────────────────────────────────────────────────────────────
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)

# ─── Exit Thresholds ──────────────────────────────────────────────────────────
HARD_STOP_PCT          = 0.10    # 10% max loss from entry
FLASH_CRASH_PCT        = 0.05    # 5% drop in one interval → emergency

# ─── Partial Profit ───────────────────────────────────────────────────────────
PARTIAL_PROFIT_PCT     = 0.50    # Take partial at +50%
PARTIAL_SELL_FRACTION  = 0.50    # Sell 50% of position

# ─── Tiered Trailing Stops (for runner after partial) ─────────────────────────
TRAILING_TIERS = [
    (1.50, 0.08),   # +150%+ profit → trail 8% from high
    (1.00, 0.10),   # +100%+ profit → trail 10%
    (0.75, 0.12),   # +75%+  profit → trail 12%
    (0.50, 0.15),   # +50%+  profit → trail 15%
]
DEFAULT_TRAILING_PCT   = 0.20    # Before +50%, if somehow needed: 20%

# ─── Time Decay ───────────────────────────────────────────────────────────────
STALE_TRADE_HOURS      = 2      # Exit if position stagnates after this long
STALE_TRADE_MIN_PNL    = -0.05  # … and P&L is between this …
STALE_TRADE_MAX_PNL    = 0.10   # … and this (going nowhere)

# ─── Technical Exit Signals ───────────────────────────────────────────────────
EMA_FAST_PERIOD        = 9
EMA_SLOW_PERIOD        = 21

# ─── Re-entry Cooldown ───────────────────────────────────────────────────────
REENTRY_COOLDOWN_MIN   = 30     # Minutes to wait before re-entering same symbol

# ─── Monitoring ───────────────────────────────────────────────────────────────
MONITOR_INTERVAL       = 30     # seconds between checks
EMERGENCY_TIMEOUT      = 5      # seconds to wait for limit fill before market

# ─── MongoDB ──────────────────────────────────────────────────────────────────
mongo_client       = MongoClient("mongodb://localhost:27017/")
db                 = mongo_client["breakout_db"]
trades_collection  = db["trades_v2"]
cooldown_collection = db["cooldowns"]


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING
# ══════════════════════════════════════════════════════════════════════════════

class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh  = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh  = logging.FileHandler("sell_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("sell_bot")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


# ══════════════════════════════════════════════════════════════════════════════
# IBKR CLIENT
# ══════════════════════════════════════════════════════════════════════════════

class IBKRSellClient:
    def __init__(self):
        util.startLoop()
        self.ib = IB()

    def connect(self):
        self.ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
        log.info(f"IBKR connected | accounts: {self.ib.wrapper.accounts}")

    def disconnect(self):
        self.ib.disconnect()

    # ── Price helpers ─────────────────────────────────────────────────────────

    def get_last_price(self, symbol: str) -> Optional[float]:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "106", False, False)
        self.ib.sleep(3)
        self.ib.cancelMktData(contract)

        for label, val in [("last", ticker.last), ("ask", ticker.ask),
                           ("bid", ticker.bid), ("close", ticker.close)]:
            if val and float(val) > 0:
                log.debug(f"{symbol}: price source='{label}' ${float(val):.4f}")
                return float(val)

        log.warning(f"{symbol}: no price available")
        return None

    def get_ticker(self, symbol: str) -> Ticker:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "", False, False)
        self.ib.sleep(2)
        return ticker

    # ── Historical bars ───────────────────────────────────────────────────────

    def get_bars(self, symbol: str, n: int = 70) -> Optional[pd.DataFrame]:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)

        for what_to_show in ("TRADES", "MIDPOINT"):
            raw = self.ib.reqHistoricalData(
                contract,
                endDateTime="",
                durationStr="10 D",
                barSizeSetting="5 mins",
                whatToShow=what_to_show,
                useRTH=False,
                keepUpToDate=False,
            )
            if raw:
                df = pd.DataFrame([
                    {"open": b.open, "high": b.high, "low": b.low,
                     "close": b.close, "volume": getattr(b, "volume", 0)}
                    for b in raw
                ])
                return df.tail(n).reset_index(drop=True)

        log.warning(f"{symbol}: no bars returned")
        return None

    # ── Technical indicators ──────────────────────────────────────────────────

    def compute_exit_signals(self, symbol: str) -> dict:
        """
        Compute PSAR, EMA 9, EMA 21 for exit decisions.
        Returns dict with signal flags or empty dict on failure.
        """
        df = self.get_bars(symbol, n=70)
        if df is None or len(df) < 25:
            log.warning(f"{symbol}: not enough bars for exit signals")
            return {}

        # Parabolic SAR
        psar_vals = self._compute_psar(df["high"].values, df["low"].values,
                                       df["close"].values)

        # EMA 9 / 21
        ema_fast = df["close"].ewm(span=EMA_FAST_PERIOD, adjust=False).mean()
        ema_slow = df["close"].ewm(span=EMA_SLOW_PERIOD, adjust=False).mean()

        price     = float(df["close"].iloc[-1])
        psar_now  = float(psar_vals[-1])
        ema9_now  = float(ema_fast.iloc[-1])
        ema21_now = float(ema_slow.iloc[-1])

        # Previous bar for crossover detection
        ema9_prev  = float(ema_fast.iloc[-2])
        ema21_prev = float(ema_slow.iloc[-2])

        psar_bearish       = psar_now > price       # SAR above price = downtrend
        ema_death_cross    = (ema9_prev >= ema21_prev) and (ema9_now < ema21_now)
        ema_bearish        = ema9_now < ema21_now
        price_below_ema9   = price < ema9_now
        price_below_ema21  = price < ema21_now

        signals = {
            "price":              price,
            "psar":               psar_now,
            "ema_9":              ema9_now,
            "ema_21":             ema21_now,
            "psar_bearish":       psar_bearish,
            "ema_death_cross":    ema_death_cross,
            "ema_bearish":        ema_bearish,
            "price_below_ema9":   price_below_ema9,
            "price_below_ema21":  price_below_ema21,
        }

        log.info(
            f"{symbol} signals: PSAR={psar_now:.4f}({'BEAR' if psar_bearish else 'BULL'}) "
            f"EMA9={ema9_now:.4f} EMA21={ema21_now:.4f} "
            f"cross={'YES' if ema_death_cross else 'no'} "
            f"below9={'YES' if price_below_ema9 else 'no'} "
            f"below21={'YES' if price_below_ema21 else 'no'}"
        )
        return signals

    @staticmethod
    def _compute_psar(highs, lows, closes, iaf=0.02, max_af=0.2) -> np.ndarray:
        n    = len(closes)
        psar = closes.copy().astype(float)
        bull = True
        af   = iaf
        hp   = float(highs[0])
        lp   = float(lows[0])

        for i in range(2, n):
            if bull:
                psar[i] = psar[i - 1] + af * (hp - psar[i - 1])
                psar[i] = min(psar[i], lows[i - 1], lows[i - 2])
                if lows[i] < psar[i]:
                    bull    = False
                    psar[i] = hp
                    lp      = lows[i]
                    af      = iaf
                else:
                    if highs[i] > hp:
                        hp = highs[i]
                        af = min(af + iaf, max_af)
            else:
                psar[i] = psar[i - 1] + af * (lp - psar[i - 1])
                psar[i] = max(psar[i], highs[i - 1], highs[i - 2])
                if highs[i] > psar[i]:
                    bull    = True
                    psar[i] = lp
                    hp      = highs[i]
                    af      = iaf
                else:
                    if lows[i] < lp:
                        lp = lows[i]
                        af = min(af + iaf, max_af)
        return psar

    # ── Order execution ───────────────────────────────────────────────────────

    def get_avg_fill_price_from_broker(self, symbol: str, order_id: int) -> Optional[float]:
        for trade in self.ib.trades():
            if (trade.contract.symbol == symbol
                    and trade.order.orderId == order_id
                    and trade.orderStatus.avgFillPrice > 0):
                return float(trade.orderStatus.avgFillPrice)

        fills   = self.ib.fills()
        matched = [f for f in fills
                   if f.contract.symbol == symbol and f.execution.orderId == order_id]
        if matched:
            total_qty  = sum(f.execution.shares for f in matched)
            total_cost = sum(f.execution.shares * f.execution.price for f in matched)
            return total_cost / total_qty

        return None

    def place_limit_sell(self, symbol: str, price: float, qty: int,
                         emergency: bool = False) -> Trade:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)

        if emergency:
            price = price * 0.99  # 1% below for faster fill

        order = LimitOrder("SELL", qty, round(price, 2))
        order.outsideRth = True
        trade = self.ib.placeOrder(contract, order)
        self.ib.sleep(1)

        log.info(f"SELL LIMIT {symbol} x{qty} @ ${price:.2f} "
                 f"orderId={trade.order.orderId} emergency={emergency}")
        return trade

    def place_market_sell(self, symbol: str, qty: int) -> Trade:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        order = MarketOrder("SELL", qty)
        trade = self.ib.placeOrder(contract, order)
        self.ib.sleep(1)
        log.info(f"SELL MARKET {symbol} x{qty} orderId={trade.order.orderId}")
        return trade

    def get_sell_fill_price(self, symbol: str, sell_order_id: int,
                            fallback_price: float) -> float:
        self.ib.sleep(3)
        avg = self.get_avg_fill_price_from_broker(symbol, sell_order_id)
        if avg:
            return avg
        log.warning(f"{symbol}: sell fill not confirmed, using fallback ${fallback_price:.4f}")
        return fallback_price


# ══════════════════════════════════════════════════════════════════════════════
# MONGODB HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def get_open_trades() -> list:
    return list(trades_collection.find({"status": "open"}))


def get_effective_entry_price(trade: dict, ibkr: IBKRSellClient) -> float:
    """Priority: avg_fill_price (DB) → broker lookup → entry_price (DB)."""
    if trade.get("avg_fill_price"):
        return float(trade["avg_fill_price"])

    order_id = trade.get("order_id")
    symbol   = trade.get("instrument") or trade.get("symbol")

    if order_id:
        broker_avg = ibkr.get_avg_fill_price_from_broker(symbol, order_id)
        if broker_avg:
            trades_collection.update_one(
                {"_id": trade["_id"]},
                {"$set": {"avg_fill_price": broker_avg, "entry_price": broker_avg}},
            )
            return broker_avg

    return float(trade["entry_price"])


def update_trade_field(trade_id, updates: dict):
    updates["updated_at"] = datetime.now(timezone.utc)
    trades_collection.update_one({"_id": trade_id}, {"$set": updates})


def close_trade(trade: dict, exit_price: float, reason: str, qty_sold: int = None):
    """Close (or partially close) a trade in MongoDB."""
    symbol      = trade.get("instrument") or trade.get("symbol")
    entry_price = float(trade.get("avg_fill_price") or trade["entry_price"])
    total_qty   = trade.get("quantity") or trade.get("qty")
    pnl_pct     = (exit_price - entry_price) / entry_price

    if qty_sold is None:
        qty_sold = total_qty

    realized_pnl = (exit_price - entry_price) * qty_sold

    if qty_sold >= total_qty:
        # Full close
        trades_collection.update_one(
            {"_id": trade["_id"]},
            {"$set": {
                "status":           "closed",
                "exit_price":       exit_price,
                "exit_timestamp":   datetime.now(timezone.utc),
                "reason":           reason,
                "pnl_pct":          round(pnl_pct * 100, 2),
                "realized_pnl":     realized_pnl,
                "updated_at":       datetime.now(timezone.utc),
            }},
        )
        log.info(
            f"CLOSED {symbol} | entry=${entry_price:.4f} exit=${exit_price:.4f} "
            f"pnl={pnl_pct * 100:+.2f}% | ${realized_pnl:+.2f} | reason={reason}"
        )
        set_cooldown(symbol, reason)
    else:
        # Partial close — reduce quantity, record partial exit
        remaining = total_qty - qty_sold
        trades_collection.update_one(
            {"_id": trade["_id"]},
            {"$set": {
                "quantity":              remaining,
                "partial_exit_price":    exit_price,
                "partial_exit_qty":      qty_sold,
                "partial_exit_time":     datetime.now(timezone.utc),
                "partial_exit_reason":   reason,
                "partial_realized_pnl":  realized_pnl,
                "phase":                 "runner",
                "updated_at":            datetime.now(timezone.utc),
            }},
        )
        log.info(
            f"PARTIAL CLOSE {symbol} | sold {qty_sold}/{total_qty} @ ${exit_price:.4f} "
            f"pnl={pnl_pct * 100:+.2f}% | ${realized_pnl:+.2f} | "
            f"remaining={remaining} shares → runner mode"
        )


# ══════════════════════════════════════════════════════════════════════════════
# POSITION CONSOLIDATION
# ══════════════════════════════════════════════════════════════════════════════

def consolidate_duplicate_positions(open_trades: list) -> list:
    """
    If the buy bot opened multiple trades for the same symbol,
    merge them into a single trade record in MongoDB so the exit
    logic operates on the full position size.

    Returns the deduplicated list of trades.
    """
    by_symbol = defaultdict(list)
    for t in open_trades:
        sym = t.get("instrument") or t.get("symbol")
        if sym:
            by_symbol[sym].append(t)

    consolidated = []
    for sym, trades in by_symbol.items():
        if len(trades) == 1:
            consolidated.append(trades[0])
            continue

        log.warning(
            f"{sym}: CONSOLIDATING {len(trades)} duplicate open trades into 1"
        )

        total_qty   = 0
        total_cost  = 0.0
        earliest_ts = None
        highest     = 0.0
        keep_trade  = None

        for t in trades:
            qty   = t.get("quantity") or t.get("qty") or 0
            entry = float(t.get("avg_fill_price") or t.get("entry_price") or 0)
            total_qty  += qty
            total_cost += qty * entry

            ts = t.get("entry_timestamp")
            if ts and (earliest_ts is None or ts < earliest_ts):
                earliest_ts = ts

            h = t.get("highest_price", entry)
            if h > highest:
                highest = h

            if keep_trade is None:
                keep_trade = t
            else:
                trades_collection.update_one(
                    {"_id": t["_id"]},
                    {"$set": {
                        "status":     "merged",
                        "merged_into": keep_trade["_id"],
                        "updated_at": datetime.now(timezone.utc),
                    }},
                )
                log.info(f"{sym}: merged trade {t['_id']} into {keep_trade['_id']}")

        avg_entry = total_cost / total_qty if total_qty > 0 else 0
        updates = {
            "quantity":           total_qty,
            "avg_fill_price":     avg_entry,
            "entry_price":        avg_entry,
            "highest_price":      highest,
            "consolidated":       True,
            "consolidated_count": len(trades),
        }
        if earliest_ts:
            updates["entry_timestamp"] = earliest_ts

        update_trade_field(keep_trade["_id"], updates)

        keep_trade["quantity"]       = total_qty
        keep_trade["avg_fill_price"] = avg_entry
        keep_trade["entry_price"]    = avg_entry
        keep_trade["highest_price"]  = highest

        log.info(
            f"{sym}: CONSOLIDATED → {total_qty} shares @ avg ${avg_entry:.4f} "
            f"(was {len(trades)} separate trades)"
        )
        consolidated.append(keep_trade)

    return consolidated


# ══════════════════════════════════════════════════════════════════════════════
# RE-ENTRY COOLDOWN
# ══════════════════════════════════════════════════════════════════════════════

def set_cooldown(symbol: str, reason: str):
    expires_at = datetime.now(timezone.utc) + timedelta(minutes=REENTRY_COOLDOWN_MIN)
    cooldown_collection.update_one(
        {"symbol": symbol},
        {"$set": {
            "symbol":       symbol,
            "reason":       reason,
            "closed_at":    datetime.now(timezone.utc),
            "expires_at":   expires_at,
            "cooldown_min": REENTRY_COOLDOWN_MIN,
        }},
        upsert=True,
    )
    log.info(
        f"{symbol}: COOLDOWN SET — no re-entry for {REENTRY_COOLDOWN_MIN} min "
        f"(until {expires_at.strftime('%H:%M:%S')} UTC) | close reason: {reason}"
    )


def is_in_cooldown(symbol: str) -> bool:
    record = cooldown_collection.find_one({"symbol": symbol})
    if not record:
        return False

    expires_at = record.get("expires_at")
    if not expires_at:
        return False

    now = datetime.now(timezone.utc)
    if expires_at.tzinfo is None:
        in_cooldown = datetime.now() < expires_at
    else:
        in_cooldown = now < expires_at

    if in_cooldown:
        remaining = (expires_at - now).total_seconds() / 60
        log.info(f"{symbol}: IN COOLDOWN — {remaining:.1f} min remaining "
                 f"(closed for: {record.get('reason', 'unknown')})")
    else:
        cooldown_collection.delete_one({"symbol": symbol})

    return in_cooldown


def clear_cooldown(symbol: str):
    cooldown_collection.delete_one({"symbol": symbol})
    log.info(f"{symbol}: cooldown manually cleared")


# ══════════════════════════════════════════════════════════════════════════════
# RISK MANAGEMENT
# ══════════════════════════════════════════════════════════════════════════════

def get_trailing_stop_pct(pnl_pct: float) -> float:
    for threshold, trail in TRAILING_TIERS:
        if pnl_pct >= threshold:
            return trail
    return DEFAULT_TRAILING_PCT


def compute_trailing_stop(entry_price: float, highest_price: float,
                          current_pnl_pct: float) -> float:
    trail_pct  = get_trailing_stop_pct(current_pnl_pct)
    trail_stop = highest_price * (1 - trail_pct)
    hard_stop  = entry_price * (1 - HARD_STOP_PCT)
    return max(trail_stop, hard_stop)


def check_flash_crash(symbol: str, current_price: float, trade: dict) -> Optional[str]:
    """Detect sudden price drops within a single monitoring interval."""
    last_price = trade.get("last_price")

    update_trade_field(trade["_id"], {
        "last_price":      current_price,
        "last_price_time": datetime.now(timezone.utc),
    })

    if last_price and last_price > 0:
        drop_pct = (current_price - last_price) / last_price
        if drop_pct <= -FLASH_CRASH_PCT:
            log.warning(f"{symbol}: FLASH CRASH {drop_pct * 100:.2f}% in {MONITOR_INTERVAL}s")
            return f"flash_crash_{abs(drop_pct) * 100:.1f}pct"

    return None


def check_stale_trade(trade: dict, pnl_pct: float) -> bool:
    entry_time = trade.get("entry_timestamp")
    if not entry_time:
        return False

    if entry_time.tzinfo is None:
        hours_held = (datetime.now() - entry_time).total_seconds() / 3600
    else:
        hours_held = (datetime.now(timezone.utc) - entry_time).total_seconds() / 3600

    if hours_held >= STALE_TRADE_HOURS:
        if STALE_TRADE_MIN_PNL <= pnl_pct <= STALE_TRADE_MAX_PNL:
            log.info(
                f"{trade.get('instrument') or trade.get('symbol')}: "
                f"stale trade — {hours_held:.1f}h held, pnl={pnl_pct * 100:+.2f}% "
                f"(flat zone {STALE_TRADE_MIN_PNL * 100}% to {STALE_TRADE_MAX_PNL * 100}%)"
            )
            return True
    return False


# ══════════════════════════════════════════════════════════════════════════════
# EMERGENCY EXIT
# ══════════════════════════════════════════════════════════════════════════════

def emergency_exit(ibkr: IBKRSellClient, symbol: str, qty: int,
                   reason: str) -> tuple[float, str]:
    """Aggressive limit → market fallback for urgent exits."""
    log.warning(f"{symbol}: EMERGENCY EXIT — {reason}")

    ticker      = ibkr.get_ticker(symbol)
    current_bid = ticker.bid or ticker.last or ticker.close

    if not current_bid:
        log.error(f"{symbol}: no price for emergency exit!")
        return 0.0, "exit_failed_no_price"

    trade    = ibkr.place_limit_sell(symbol, current_bid, qty, emergency=True)
    order_id = trade.order.orderId

    for _ in range(EMERGENCY_TIMEOUT):
        ibkr.ib.sleep(1)
        fill = ibkr.get_avg_fill_price_from_broker(symbol, order_id)
        if fill:
            log.info(f"{symbol}: emergency limit filled @ ${fill:.4f}")
            return fill, reason

    log.warning(f"{symbol}: limit not filled in {EMERGENCY_TIMEOUT}s → MARKET")
    try:
        ibkr.ib.cancelOrder(trade.order)
    except Exception:
        pass

    market_trade = ibkr.place_market_sell(symbol, qty)
    ibkr.ib.sleep(2)
    fill = ibkr.get_avg_fill_price_from_broker(symbol, market_trade.order.orderId)
    if fill:
        return fill, f"{reason}_market"

    return current_bid * 0.98, f"{reason}_estimated"


# ══════════════════════════════════════════════════════════════════════════════
# EXIT DECISION ENGINE
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class ExitDecision:
    action: str            # "hold", "partial_sell", "full_sell", "emergency_sell"
    reason: str
    qty_to_sell: int = 0


def evaluate_position(trade: dict, current_price: float, entry_price: float,
                      ibkr: IBKRSellClient) -> ExitDecision:
    """
    Master exit logic — evaluates all conditions in priority order.

    v2.3 CHANGES:
    - Liquidity crisis check REMOVED entirely
    - Trailing stop confirmation requires PSAR bearish ONLY
    - When profitable, only PSAR-based signals trigger exits
    """
    symbol    = trade.get("instrument") or trade.get("symbol")
    total_qty = trade.get("quantity") or trade.get("qty")
    highest   = trade.get("highest_price", entry_price)
    phase     = trade.get("phase", "initial")
    pnl_pct   = (current_price - entry_price) / entry_price

    # Track new high
    if current_price > highest:
        highest = current_price
        update_trade_field(trade["_id"], {"highest_price": highest})

    drawdown_from_high = (current_price - highest) / highest if highest > 0 else 0

    log.info(
        f"{symbol}: phase={phase} | entry=${entry_price:.4f} now=${current_price:.4f} "
        f"pnl={pnl_pct * 100:+.2f}% | high=${highest:.4f} dd={drawdown_from_high * 100:.2f}% "
        f"qty={total_qty}"
    )

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 1 — Hard stop (always active, loss protection)
    # ══════════════════════════════════════════════════════════════════════
    if pnl_pct <= -HARD_STOP_PCT:
        return ExitDecision("emergency_sell", "hard_stop_10pct", total_qty)

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 2 — Flash crash (always active)
    # ══════════════════════════════════════════════════════════════════════
    flash = check_flash_crash(symbol, current_price, trade)
    if flash:
        return ExitDecision("emergency_sell", flash, total_qty)

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 3 — Partial profit (only in "initial" phase)
    # ══════════════════════════════════════════════════════════════════════
    if phase == "initial" and pnl_pct >= PARTIAL_PROFIT_PCT:
        sell_qty = max(1, int(total_qty * PARTIAL_SELL_FRACTION))
        if sell_qty >= total_qty:
            return ExitDecision("full_sell", f"profit_target_{pnl_pct * 100:.1f}pct", total_qty)
        return ExitDecision("partial_sell", f"partial_profit_{pnl_pct * 100:.1f}pct", sell_qty)

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 4 — Trailing stop (PSAR-confirmed ONLY)
    #   The trailing stop level is computed from tiers, but it ONLY
    #   triggers if PSAR is bearish. This keeps you in strong uptrends
    #   that pull back but maintain bullish PSAR structure.
    # ══════════════════════════════════════════════════════════════════════
    stop_level = compute_trailing_stop(entry_price, highest, pnl_pct)

    if stop_level > entry_price * (1 - HARD_STOP_PCT):
        if current_price <= stop_level:
            signals = ibkr.compute_exit_signals(symbol)
            psar_confirmed = False

            if signals:
                psar_bearish = signals.get("psar_bearish", False)

                if psar_bearish:
                    psar_confirmed = True
                    log.info(
                        f"{symbol}: trailing stop CONFIRMED — PSAR bearish "
                        f"(PSAR={signals.get('psar', 0):.4f} > price={current_price:.4f})"
                    )
                else:
                    log.info(
                        f"{symbol}: trailing stop level breached (${stop_level:.4f}) "
                        f"but PSAR still bullish — HOLDING through pullback"
                    )
            else:
                # No signal data (bars failed) — trigger stop as safety fallback
                psar_confirmed = True
                log.warning(
                    f"{symbol}: trailing stop breached & no signal data — "
                    f"triggering stop as safety fallback"
                )

            if psar_confirmed:
                trail_pct = get_trailing_stop_pct(pnl_pct)
                return ExitDecision(
                    "full_sell",
                    f"trailing_stop_{trail_pct * 100:.0f}pct_psar_confirmed",
                    total_qty,
                )

    update_trade_field(trade["_id"], {"trailing_stop": stop_level})

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 5 — Technical exit signals (runner phase)
    #   PSAR bearish + price below EMA 9 → exit
    #   EMA 9 death-crosses EMA 21       → exit
    # ══════════════════════════════════════════════════════════════════════
    if phase == "runner":
        signals = ibkr.compute_exit_signals(symbol)
        if signals:
            if signals.get("psar_bearish") and signals.get("price_below_ema9"):
                return ExitDecision(
                    "full_sell",
                    "psar_bearish_below_ema9",
                    total_qty,
                )
            if signals.get("ema_death_cross"):
                return ExitDecision(
                    "full_sell",
                    "ema_9_21_death_cross",
                    total_qty,
                )

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 6 — Stale trade (going nowhere for too long)
    # ══════════════════════════════════════════════════════════════════════
    if check_stale_trade(trade, pnl_pct):
        return ExitDecision("full_sell", "stale_trade_timeout", total_qty)

    # ══════════════════════════════════════════════════════════════════════
    # DEFAULT — Hold
    # ══════════════════════════════════════════════════════════════════════
    trail_pct = get_trailing_stop_pct(pnl_pct)
    log.info(
        f"{symbol}: HOLD | trail={trail_pct * 100:.0f}% "
        f"stop=${stop_level:.4f} | phase={phase}"
    )
    return ExitDecision("hold", "holding", 0)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN MONITOR LOOP
# ══════════════════════════════════════════════════════════════════════════════

def monitor_positions(ibkr: IBKRSellClient):
    log.info("=" * 60)
    log.info("SELL BOT v2.3 — Smart Position Manager")
    log.info("  Liquidity crisis: DISABLED (removed)")
    log.info("  Exit confirmation: PSAR bearish only")
    log.info("  Duplicate positions: auto-consolidated")
    log.info(f"Hard Stop: {HARD_STOP_PCT * 100}% | "
             f"Partial: {PARTIAL_PROFIT_PCT * 100}% ({PARTIAL_SELL_FRACTION * 100}% of qty) | "
             f"Flash: {FLASH_CRASH_PCT * 100}%")
    log.info(f"Trailing tiers: {[(f'+{t*100:.0f}%', f'{p*100:.0f}%') for t, p in TRAILING_TIERS]}")
    log.info(f"Trail confirmation: PSAR bearish ONLY")
    log.info(f"Re-entry cooldown: {REENTRY_COOLDOWN_MIN} min")
    log.info(f"Stale exit: >{STALE_TRADE_HOURS}h if PnL in "
             f"[{STALE_TRADE_MIN_PNL * 100}%, {STALE_TRADE_MAX_PNL * 100}%]")
    log.info("=" * 60)

    while True:
        try:
            open_trades = get_open_trades()

            # ── Consolidate duplicate positions before processing ─────────
            open_trades = consolidate_duplicate_positions(open_trades)

            log.info(f"--- Checking {len(open_trades)} open position(s) ---")

            for trade in open_trades:
                symbol    = trade.get("instrument") or trade.get("symbol")
                total_qty = trade.get("quantity") or trade.get("qty")

                if not symbol or not total_qty:
                    log.warning(f"Skipping trade {trade['_id']}: missing symbol or qty")
                    continue

                entry_price   = get_effective_entry_price(trade, ibkr)
                current_price = ibkr.get_last_price(symbol)
                if not current_price:
                    log.warning(f"{symbol}: no current price, skipping")
                    continue

                decision = evaluate_position(trade, current_price, entry_price, ibkr)

                if decision.action == "hold":
                    continue

                # ── Execute the decision ──────────────────────────────────

                if decision.action == "emergency_sell":
                    exit_price, final_reason = emergency_exit(
                        ibkr, symbol, decision.qty_to_sell, decision.reason
                    )
                    close_trade(trade, exit_price, final_reason, decision.qty_to_sell)

                elif decision.action == "partial_sell":
                    sell_trade    = ibkr.place_limit_sell(symbol, current_price,
                                                          decision.qty_to_sell)
                    sell_order_id = sell_trade.order.orderId
                    confirmed     = ibkr.get_sell_fill_price(symbol, sell_order_id,
                                                               current_price)
                    close_trade(trade, confirmed, decision.reason, decision.qty_to_sell)

                elif decision.action == "full_sell":
                    sell_trade    = ibkr.place_limit_sell(symbol, current_price,
                                                          decision.qty_to_sell)
                    sell_order_id = sell_trade.order.orderId
                    confirmed     = ibkr.get_sell_fill_price(symbol, sell_order_id,
                                                               current_price)
                    close_trade(trade, confirmed, decision.reason, decision.qty_to_sell)

        except Exception as e:
            log.error(f"Monitor error: {e}", exc_info=True)

        time.sleep(MONITOR_INTERVAL)


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

def main():
    log.info("=== Sell Bot v2.3 Starting ===")
    ibkr = IBKRSellClient()

    try:
        ibkr.connect()
        monitor_positions(ibkr)
    except KeyboardInterrupt:
        log.info("Sell bot stopped by user.")
    except Exception as e:
        log.error(f"Fatal error: {e}", exc_info=True)
    finally:
        ibkr.disconnect()
        log.info("IBKR disconnected.")


if __name__ == "__main__":
    main()


In [ ]:
"""
Sell Bot v2.3 — Smart Position Manager
═══════════════════════════════════════
Monitors open positions via IBKR + MongoDB.

CHANGES from v2.1:
─────────────────
1. REMOVED liquidity crisis check entirely (was causing false exits on
   low-priced stocks where wide spreads are normal).
2. Profitable exits now rely on PSAR confirmation — trailing stop only
   triggers when PSAR is bearish (not EMA21 alone).
3. Duplicate position consolidation added — merges multiple open trades
   for the same symbol into one so exit logic works on full position.

EXIT STRATEGY (designed for momentum breakout plays):
─────────────────────────────────────────────────────
Phase 1 — PROTECTION (from entry until profitable):
  • Hard stop: -10% from entry
  • Flash crash: -5% drop in a single 30-second interval → emergency exit

Phase 2 — PARTIAL PROFIT at +50%:
  • Sell 50% of position, lock in gains
  • Remaining 50% enters "runner" mode with trailing stop

Phase 3 — RUNNER (remaining shares after partial):
  • Trailing stop tightens in tiers based on how far price has run:
      +50% to +75%  →  trail 15% from high
      +75% to +100% →  trail 12% from high
      +100% to +150% → trail 10% from high
      +150%+         → trail  8% from high
  • TRAILING STOP CONFIRMATION: trail stop only triggers if PSAR is bearish
    (prevents exiting on normal pullbacks in a trend)
  • PSAR flip bearish + price below EMA 9 → exit runner
  • EMA 9 crosses below EMA 21 → exit runner (trend is done)

Phase 4 — TIME DECAY (if stock goes nowhere):
  • If position is between -5% and +10% after 2 hours → exit

EMERGENCY EXITS (always active, any phase):
  • Hard stop (-10%)
  • Flash crash detection

RE-ENTRY COOLDOWN:
  • After closing a position, a cooldown period is recorded in MongoDB
  • Buy bot should check cooldown before re-entering the same symbol
"""

import logging
import sys
import time
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta
import random
from typing import Optional
from dataclasses import dataclass
from collections import defaultdict

from pymongo import MongoClient
from ib_insync import *

import warnings
warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# ─── IBKR ─────────────────────────────────────────────────────────────────────
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)

# ─── Exit Thresholds ──────────────────────────────────────────────────────────
HARD_STOP_PCT          = 0.10    # 10% max loss from entry
FLASH_CRASH_PCT        = 0.05    # 5% drop in one interval → emergency

# ─── Partial Profit ───────────────────────────────────────────────────────────
PARTIAL_PROFIT_PCT     = 0.50    # Take partial at +50%
PARTIAL_SELL_FRACTION  = 0.50    # Sell 50% of position

# ─── Tiered Trailing Stops (for runner after partial) ─────────────────────────
TRAILING_TIERS = [
    (1.50, 0.08),   # +150%+ profit → trail 8% from high
    (1.00, 0.10),   # +100%+ profit → trail 10%
    (0.75, 0.12),   # +75%+  profit → trail 12%
    (0.50, 0.15),   # +50%+  profit → trail 15%
]
DEFAULT_TRAILING_PCT   = 0.20    # Before +50%, if somehow needed: 20%

# ─── Time Decay ───────────────────────────────────────────────────────────────
STALE_TRADE_HOURS      = 2      # Exit if position stagnates after this long
STALE_TRADE_MIN_PNL    = -0.05  # … and P&L is between this …
STALE_TRADE_MAX_PNL    = 0.10   # … and this (going nowhere)

# ─── Technical Exit Signals ───────────────────────────────────────────────────
EMA_FAST_PERIOD        = 9
EMA_SLOW_PERIOD        = 21

# ─── Re-entry Cooldown ───────────────────────────────────────────────────────
REENTRY_COOLDOWN_MIN   = 30     # Minutes to wait before re-entering same symbol

# ─── Monitoring ───────────────────────────────────────────────────────────────
MONITOR_INTERVAL       = 30     # seconds between checks
EMERGENCY_TIMEOUT      = 5      # seconds to wait for limit fill before market

# ─── MongoDB ──────────────────────────────────────────────────────────────────
mongo_client       = MongoClient("mongodb://localhost:27017/")
db                 = mongo_client["breakout_db"]
trades_collection  = db["trades_v2"]
cooldown_collection = db["cooldowns"]


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING
# ══════════════════════════════════════════════════════════════════════════════

class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh  = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh  = logging.FileHandler("sell_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("sell_bot")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


# ══════════════════════════════════════════════════════════════════════════════
# IBKR CLIENT
# ══════════════════════════════════════════════════════════════════════════════

class IBKRSellClient:
    def __init__(self):
        util.startLoop()
        self.ib = IB()

    def connect(self):
        self.ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
        log.info(f"IBKR connected | accounts: {self.ib.wrapper.accounts}")

    def disconnect(self):
        self.ib.disconnect()

    # ── Price helpers ─────────────────────────────────────────────────────────

    def get_last_price(self, symbol: str) -> Optional[float]:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "106", False, False)
        self.ib.sleep(3)
        self.ib.cancelMktData(contract)

        for label, val in [("last", ticker.last), ("ask", ticker.ask),
                           ("bid", ticker.bid), ("close", ticker.close)]:
            if val and float(val) > 0:
                log.debug(f"{symbol}: price source='{label}' ${float(val):.4f}")
                return float(val)

        log.warning(f"{symbol}: no price available")
        return None

    def get_ticker(self, symbol: str) -> Ticker:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "", False, False)
        self.ib.sleep(2)
        return ticker

    # ── Historical bars ───────────────────────────────────────────────────────

    def get_bars(self, symbol: str, n: int = 70) -> Optional[pd.DataFrame]:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)

        for what_to_show in ("TRADES", "MIDPOINT"):
            raw = self.ib.reqHistoricalData(
                contract,
                endDateTime="",
                durationStr="10 D",
                barSizeSetting="5 mins",
                whatToShow=what_to_show,
                useRTH=False,
                keepUpToDate=False,
            )
            if raw:
                df = pd.DataFrame([
                    {"open": b.open, "high": b.high, "low": b.low,
                     "close": b.close, "volume": getattr(b, "volume", 0)}
                    for b in raw
                ])
                return df.tail(n).reset_index(drop=True)

        log.warning(f"{symbol}: no bars returned")
        return None

    # ── Technical indicators ──────────────────────────────────────────────────

    def compute_exit_signals(self, symbol: str) -> dict:
        """
        Compute PSAR, EMA 9, EMA 21 for exit decisions.
        Returns dict with signal flags or empty dict on failure.
        """
        df = self.get_bars(symbol, n=70)
        if df is None or len(df) < 25:
            log.warning(f"{symbol}: not enough bars for exit signals")
            return {}

        # Parabolic SAR
        psar_vals = self._compute_psar(df["high"].values, df["low"].values,
                                       df["close"].values)

        # EMA 9 / 21
        ema_fast = df["close"].ewm(span=EMA_FAST_PERIOD, adjust=False).mean()
        ema_slow = df["close"].ewm(span=EMA_SLOW_PERIOD, adjust=False).mean()

        price     = float(df["close"].iloc[-1])
        psar_now  = float(psar_vals[-1])
        ema9_now  = float(ema_fast.iloc[-1])
        ema21_now = float(ema_slow.iloc[-1])

        # Previous bar for crossover detection
        ema9_prev  = float(ema_fast.iloc[-2])
        ema21_prev = float(ema_slow.iloc[-2])

        psar_bearish       = psar_now > price       # SAR above price = downtrend
        ema_death_cross    = (ema9_prev >= ema21_prev) and (ema9_now < ema21_now)
        ema_bearish        = ema9_now < ema21_now
        price_below_ema9   = price < ema9_now
        price_below_ema21  = price < ema21_now

        signals = {
            "price":              price,
            "psar":               psar_now,
            "ema_9":              ema9_now,
            "ema_21":             ema21_now,
            "psar_bearish":       psar_bearish,
            "ema_death_cross":    ema_death_cross,
            "ema_bearish":        ema_bearish,
            "price_below_ema9":   price_below_ema9,
            "price_below_ema21":  price_below_ema21,
        }

        log.info(
            f"{symbol} signals: PSAR={psar_now:.4f}({'BEAR' if psar_bearish else 'BULL'}) "
            f"EMA9={ema9_now:.4f} EMA21={ema21_now:.4f} "
            f"cross={'YES' if ema_death_cross else 'no'} "
            f"below9={'YES' if price_below_ema9 else 'no'} "
            f"below21={'YES' if price_below_ema21 else 'no'}"
        )
        return signals

    @staticmethod
    def _compute_psar(highs, lows, closes, iaf=0.02, max_af=0.2) -> np.ndarray:
        n    = len(closes)
        psar = closes.copy().astype(float)
        bull = True
        af   = iaf
        hp   = float(highs[0])
        lp   = float(lows[0])

        for i in range(2, n):
            if bull:
                psar[i] = psar[i - 1] + af * (hp - psar[i - 1])
                psar[i] = min(psar[i], lows[i - 1], lows[i - 2])
                if lows[i] < psar[i]:
                    bull    = False
                    psar[i] = hp
                    lp      = lows[i]
                    af      = iaf
                else:
                    if highs[i] > hp:
                        hp = highs[i]
                        af = min(af + iaf, max_af)
            else:
                psar[i] = psar[i - 1] + af * (lp - psar[i - 1])
                psar[i] = max(psar[i], highs[i - 1], highs[i - 2])
                if highs[i] > psar[i]:
                    bull    = True
                    psar[i] = lp
                    hp      = highs[i]
                    af      = iaf
                else:
                    if lows[i] < lp:
                        lp = lows[i]
                        af = min(af + iaf, max_af)
        return psar

    # ── Order execution ───────────────────────────────────────────────────────

    def get_avg_fill_price_from_broker(self, symbol: str, order_id: int) -> Optional[float]:
        for trade in self.ib.trades():
            if (trade.contract.symbol == symbol
                    and trade.order.orderId == order_id
                    and trade.orderStatus.avgFillPrice > 0):
                return float(trade.orderStatus.avgFillPrice)

        fills   = self.ib.fills()
        matched = [f for f in fills
                   if f.contract.symbol == symbol and f.execution.orderId == order_id]
        if matched:
            total_qty  = sum(f.execution.shares for f in matched)
            total_cost = sum(f.execution.shares * f.execution.price for f in matched)
            return total_cost / total_qty

        return None

    def place_limit_sell(self, symbol: str, price: float, qty: int,
                         emergency: bool = False) -> Trade:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)

        if emergency:
            price = price * 0.99  # 1% below for faster fill

        order = LimitOrder("SELL", qty, round(price, 2))
        order.outsideRth = True
        trade = self.ib.placeOrder(contract, order)
        self.ib.sleep(1)

        log.info(f"SELL LIMIT {symbol} x{qty} @ ${price:.2f} "
                 f"orderId={trade.order.orderId} emergency={emergency}")
        return trade

    def place_market_sell(self, symbol: str, qty: int) -> Trade:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        order = MarketOrder("SELL", qty)
        trade = self.ib.placeOrder(contract, order)
        self.ib.sleep(1)
        log.info(f"SELL MARKET {symbol} x{qty} orderId={trade.order.orderId}")
        return trade

    def get_sell_fill_price(self, symbol: str, sell_order_id: int,
                            fallback_price: float) -> float:
        self.ib.sleep(3)
        avg = self.get_avg_fill_price_from_broker(symbol, sell_order_id)
        if avg:
            return avg
        log.warning(f"{symbol}: sell fill not confirmed, using fallback ${fallback_price:.4f}")
        return fallback_price


# ══════════════════════════════════════════════════════════════════════════════
# MONGODB HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def get_open_trades() -> list:
    return list(trades_collection.find({"status": "open"}))


def get_effective_entry_price(trade: dict, ibkr: IBKRSellClient) -> float:
    """Priority: avg_fill_price (DB) → broker lookup → entry_price (DB)."""
    if trade.get("avg_fill_price"):
        return float(trade["avg_fill_price"])

    order_id = trade.get("order_id")
    symbol   = trade.get("instrument") or trade.get("symbol")

    if order_id:
        broker_avg = ibkr.get_avg_fill_price_from_broker(symbol, order_id)
        if broker_avg:
            trades_collection.update_one(
                {"_id": trade["_id"]},
                {"$set": {"avg_fill_price": broker_avg, "entry_price": broker_avg}},
            )
            return broker_avg

    return float(trade["entry_price"])


def update_trade_field(trade_id, updates: dict):
    updates["updated_at"] = datetime.now(timezone.utc)
    trades_collection.update_one({"_id": trade_id}, {"$set": updates})


def close_trade(trade: dict, exit_price: float, reason: str, qty_sold: int = None):
    """Close (or partially close) a trade in MongoDB."""
    symbol      = trade.get("instrument") or trade.get("symbol")
    entry_price = float(trade.get("avg_fill_price") or trade["entry_price"])
    total_qty   = trade.get("quantity") or trade.get("qty")
    pnl_pct     = (exit_price - entry_price) / entry_price

    if qty_sold is None:
        qty_sold = total_qty

    realized_pnl = (exit_price - entry_price) * qty_sold

    if qty_sold >= total_qty:
        # Full close
        trades_collection.update_one(
            {"_id": trade["_id"]},
            {"$set": {
                "status":           "closed",
                "exit_price":       exit_price,
                "exit_timestamp":   datetime.now(timezone.utc),
                "reason":           reason,
                "pnl_pct":          round(pnl_pct * 100, 2),
                "realized_pnl":     realized_pnl,
                "updated_at":       datetime.now(timezone.utc),
            }},
        )
        log.info(
            f"CLOSED {symbol} | entry=${entry_price:.4f} exit=${exit_price:.4f} "
            f"pnl={pnl_pct * 100:+.2f}% | ${realized_pnl:+.2f} | reason={reason}"
        )
        set_cooldown(symbol, reason)
    else:
        # Partial close — reduce quantity, record partial exit
        remaining = total_qty - qty_sold
        trades_collection.update_one(
            {"_id": trade["_id"]},
            {"$set": {
                "quantity":              remaining,
                "partial_exit_price":    exit_price,
                "partial_exit_qty":      qty_sold,
                "partial_exit_time":     datetime.now(timezone.utc),
                "partial_exit_reason":   reason,
                "partial_realized_pnl":  realized_pnl,
                "phase":                 "runner",
                "updated_at":            datetime.now(timezone.utc),
            }},
        )
        log.info(
            f"PARTIAL CLOSE {symbol} | sold {qty_sold}/{total_qty} @ ${exit_price:.4f} "
            f"pnl={pnl_pct * 100:+.2f}% | ${realized_pnl:+.2f} | "
            f"remaining={remaining} shares → runner mode"
        )


# ══════════════════════════════════════════════════════════════════════════════
# POSITION CONSOLIDATION
# ══════════════════════════════════════════════════════════════════════════════

def consolidate_duplicate_positions(open_trades: list) -> list:
    """
    If the buy bot opened multiple trades for the same symbol,
    merge them into a single trade record in MongoDB so the exit
    logic operates on the full position size.

    Returns the deduplicated list of trades.
    """
    by_symbol = defaultdict(list)
    for t in open_trades:
        sym = t.get("instrument") or t.get("symbol")
        if sym:
            by_symbol[sym].append(t)

    consolidated = []
    for sym, trades in by_symbol.items():
        if len(trades) == 1:
            consolidated.append(trades[0])
            continue

        log.warning(
            f"{sym}: CONSOLIDATING {len(trades)} duplicate open trades into 1"
        )

        total_qty   = 0
        total_cost  = 0.0
        earliest_ts = None
        highest     = 0.0
        keep_trade  = None

        for t in trades:
            qty   = t.get("quantity") or t.get("qty") or 0
            entry = float(t.get("avg_fill_price") or t.get("entry_price") or 0)
            total_qty  += qty
            total_cost += qty * entry

            ts = t.get("entry_timestamp")
            if ts and (earliest_ts is None or ts < earliest_ts):
                earliest_ts = ts

            h = t.get("highest_price", entry)
            if h > highest:
                highest = h

            if keep_trade is None:
                keep_trade = t
            else:
                trades_collection.update_one(
                    {"_id": t["_id"]},
                    {"$set": {
                        "status":     "merged",
                        "merged_into": keep_trade["_id"],
                        "updated_at": datetime.now(timezone.utc),
                    }},
                )
                log.info(f"{sym}: merged trade {t['_id']} into {keep_trade['_id']}")

        avg_entry = total_cost / total_qty if total_qty > 0 else 0
        updates = {
            "quantity":           total_qty,
            "avg_fill_price":     avg_entry,
            "entry_price":        avg_entry,
            "highest_price":      highest,
            "consolidated":       True,
            "consolidated_count": len(trades),
        }
        if earliest_ts:
            updates["entry_timestamp"] = earliest_ts

        update_trade_field(keep_trade["_id"], updates)

        keep_trade["quantity"]       = total_qty
        keep_trade["avg_fill_price"] = avg_entry
        keep_trade["entry_price"]    = avg_entry
        keep_trade["highest_price"]  = highest

        log.info(
            f"{sym}: CONSOLIDATED → {total_qty} shares @ avg ${avg_entry:.4f} "
            f"(was {len(trades)} separate trades)"
        )
        consolidated.append(keep_trade)

    return consolidated


# ══════════════════════════════════════════════════════════════════════════════
# RE-ENTRY COOLDOWN
# ══════════════════════════════════════════════════════════════════════════════

def set_cooldown(symbol: str, reason: str):
    expires_at = datetime.now(timezone.utc) + timedelta(minutes=REENTRY_COOLDOWN_MIN)
    cooldown_collection.update_one(
        {"symbol": symbol},
        {"$set": {
            "symbol":       symbol,
            "reason":       reason,
            "closed_at":    datetime.now(timezone.utc),
            "expires_at":   expires_at,
            "cooldown_min": REENTRY_COOLDOWN_MIN,
        }},
        upsert=True,
    )
    log.info(
        f"{symbol}: COOLDOWN SET — no re-entry for {REENTRY_COOLDOWN_MIN} min "
        f"(until {expires_at.strftime('%H:%M:%S')} UTC) | close reason: {reason}"
    )


def is_in_cooldown(symbol: str) -> bool:
    record = cooldown_collection.find_one({"symbol": symbol})
    if not record:
        return False

    expires_at = record.get("expires_at")
    if not expires_at:
        return False

    now = datetime.now(timezone.utc)
    if expires_at.tzinfo is None:
        in_cooldown = datetime.now() < expires_at
    else:
        in_cooldown = now < expires_at

    if in_cooldown:
        remaining = (expires_at - now).total_seconds() / 60
        log.info(f"{symbol}: IN COOLDOWN — {remaining:.1f} min remaining "
                 f"(closed for: {record.get('reason', 'unknown')})")
    else:
        cooldown_collection.delete_one({"symbol": symbol})

    return in_cooldown


def clear_cooldown(symbol: str):
    cooldown_collection.delete_one({"symbol": symbol})
    log.info(f"{symbol}: cooldown manually cleared")


# ══════════════════════════════════════════════════════════════════════════════
# RISK MANAGEMENT
# ══════════════════════════════════════════════════════════════════════════════

def get_trailing_stop_pct(pnl_pct: float) -> float:
    for threshold, trail in TRAILING_TIERS:
        if pnl_pct >= threshold:
            return trail
    return DEFAULT_TRAILING_PCT


def compute_trailing_stop(entry_price: float, highest_price: float,
                          current_pnl_pct: float) -> float:
    trail_pct  = get_trailing_stop_pct(current_pnl_pct)
    trail_stop = highest_price * (1 - trail_pct)
    hard_stop  = entry_price * (1 - HARD_STOP_PCT)
    return max(trail_stop, hard_stop)


def check_flash_crash(symbol: str, current_price: float, trade: dict) -> Optional[str]:
    """Detect sudden price drops within a single monitoring interval."""
    last_price = trade.get("last_price")

    update_trade_field(trade["_id"], {
        "last_price":      current_price,
        "last_price_time": datetime.now(timezone.utc),
    })

    if last_price and last_price > 0:
        drop_pct = (current_price - last_price) / last_price
        if drop_pct <= -FLASH_CRASH_PCT:
            log.warning(f"{symbol}: FLASH CRASH {drop_pct * 100:.2f}% in {MONITOR_INTERVAL}s")
            return f"flash_crash_{abs(drop_pct) * 100:.1f}pct"

    return None


def check_stale_trade(trade: dict, pnl_pct: float) -> bool:
    entry_time = trade.get("entry_timestamp")
    if not entry_time:
        return False

    if entry_time.tzinfo is None:
        hours_held = (datetime.now() - entry_time).total_seconds() / 3600
    else:
        hours_held = (datetime.now(timezone.utc) - entry_time).total_seconds() / 3600

    if hours_held >= STALE_TRADE_HOURS:
        if STALE_TRADE_MIN_PNL <= pnl_pct <= STALE_TRADE_MAX_PNL:
            log.info(
                f"{trade.get('instrument') or trade.get('symbol')}: "
                f"stale trade — {hours_held:.1f}h held, pnl={pnl_pct * 100:+.2f}% "
                f"(flat zone {STALE_TRADE_MIN_PNL * 100}% to {STALE_TRADE_MAX_PNL * 100}%)"
            )
            return True
    return False


# ══════════════════════════════════════════════════════════════════════════════
# EMERGENCY EXIT
# ══════════════════════════════════════════════════════════════════════════════

def emergency_exit(ibkr: IBKRSellClient, symbol: str, qty: int,
                   reason: str) -> tuple[float, str]:
    """Aggressive limit → market fallback for urgent exits."""
    log.warning(f"{symbol}: EMERGENCY EXIT — {reason}")

    ticker      = ibkr.get_ticker(symbol)
    current_bid = ticker.bid or ticker.last or ticker.close

    if not current_bid:
        log.error(f"{symbol}: no price for emergency exit!")
        return 0.0, "exit_failed_no_price"

    trade    = ibkr.place_limit_sell(symbol, current_bid, qty, emergency=True)
    order_id = trade.order.orderId

    for _ in range(EMERGENCY_TIMEOUT):
        ibkr.ib.sleep(1)
        fill = ibkr.get_avg_fill_price_from_broker(symbol, order_id)
        if fill:
            log.info(f"{symbol}: emergency limit filled @ ${fill:.4f}")
            return fill, reason

    log.warning(f"{symbol}: limit not filled in {EMERGENCY_TIMEOUT}s → MARKET")
    try:
        ibkr.ib.cancelOrder(trade.order)
    except Exception:
        pass

    market_trade = ibkr.place_market_sell(symbol, qty)
    ibkr.ib.sleep(2)
    fill = ibkr.get_avg_fill_price_from_broker(symbol, market_trade.order.orderId)
    if fill:
        return fill, f"{reason}_market"

    return current_bid * 0.98, f"{reason}_estimated"


# ══════════════════════════════════════════════════════════════════════════════
# EXIT DECISION ENGINE
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class ExitDecision:
    action: str            # "hold", "partial_sell", "full_sell", "emergency_sell"
    reason: str
    qty_to_sell: int = 0


def evaluate_position(trade: dict, current_price: float, entry_price: float,
                      ibkr: IBKRSellClient) -> ExitDecision:
    """
    Master exit logic — evaluates all conditions in priority order.

    v2.3 CHANGES:
    - Liquidity crisis check REMOVED entirely
    - Trailing stop confirmation requires PSAR bearish ONLY
    - When profitable, only PSAR-based signals trigger exits
    """
    symbol    = trade.get("instrument") or trade.get("symbol")
    total_qty = trade.get("quantity") or trade.get("qty")
    highest   = trade.get("highest_price", entry_price)
    phase     = trade.get("phase", "initial")
    pnl_pct   = (current_price - entry_price) / entry_price

    # Track new high
    if current_price > highest:
        highest = current_price
        update_trade_field(trade["_id"], {"highest_price": highest})

    drawdown_from_high = (current_price - highest) / highest if highest > 0 else 0

    log.info(
        f"{symbol}: phase={phase} | entry=${entry_price:.4f} now=${current_price:.4f} "
        f"pnl={pnl_pct * 100:+.2f}% | high=${highest:.4f} dd={drawdown_from_high * 100:.2f}% "
        f"qty={total_qty}"
    )

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 1 — Hard stop (always active, loss protection)
    # ══════════════════════════════════════════════════════════════════════
    if pnl_pct <= -HARD_STOP_PCT:
        return ExitDecision("emergency_sell", "hard_stop_10pct", total_qty)

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 2 — Flash crash (always active)
    # ══════════════════════════════════════════════════════════════════════
    flash = check_flash_crash(symbol, current_price, trade)
    if flash:
        return ExitDecision("emergency_sell", flash, total_qty)

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 3 — Partial profit (only in "initial" phase)
    # ══════════════════════════════════════════════════════════════════════
    if phase == "initial" and pnl_pct >= PARTIAL_PROFIT_PCT:
        sell_qty = max(1, int(total_qty * PARTIAL_SELL_FRACTION))
        if sell_qty >= total_qty:
            return ExitDecision("full_sell", f"profit_target_{pnl_pct * 100:.1f}pct", total_qty)
        return ExitDecision("partial_sell", f"partial_profit_{pnl_pct * 100:.1f}pct", sell_qty)

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 4 — Trailing stop (PSAR-confirmed ONLY)
    #   The trailing stop level is computed from tiers, but it ONLY
    #   triggers if PSAR is bearish. This keeps you in strong uptrends
    #   that pull back but maintain bullish PSAR structure.
    # ══════════════════════════════════════════════════════════════════════
    stop_level = compute_trailing_stop(entry_price, highest, pnl_pct)

    if stop_level > entry_price * (1 - HARD_STOP_PCT):
        if current_price <= stop_level:
            signals = ibkr.compute_exit_signals(symbol)
            psar_confirmed = False

            if signals:
                psar_bearish = signals.get("psar_bearish", False)

                if psar_bearish:
                    psar_confirmed = True
                    log.info(
                        f"{symbol}: trailing stop CONFIRMED — PSAR bearish "
                        f"(PSAR={signals.get('psar', 0):.4f} > price={current_price:.4f})"
                    )
                else:
                    log.info(
                        f"{symbol}: trailing stop level breached (${stop_level:.4f}) "
                        f"but PSAR still bullish — HOLDING through pullback"
                    )
            else:
                # No signal data (bars failed) — trigger stop as safety fallback
                psar_confirmed = True
                log.warning(
                    f"{symbol}: trailing stop breached & no signal data — "
                    f"triggering stop as safety fallback"
                )

            if psar_confirmed:
                trail_pct = get_trailing_stop_pct(pnl_pct)
                return ExitDecision(
                    "full_sell",
                    f"trailing_stop_{trail_pct * 100:.0f}pct_psar_confirmed",
                    total_qty,
                )

    update_trade_field(trade["_id"], {"trailing_stop": stop_level})

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 5 — Technical exit signals (runner phase)
    #   PSAR bearish + price below EMA 9 → exit
    #   EMA 9 death-crosses EMA 21       → exit
    # ══════════════════════════════════════════════════════════════════════
    if phase == "runner":
        signals = ibkr.compute_exit_signals(symbol)
        if signals:
            if signals.get("psar_bearish") and signals.get("price_below_ema9"):
                return ExitDecision(
                    "full_sell",
                    "psar_bearish_below_ema9",
                    total_qty,
                )
            if signals.get("ema_death_cross"):
                return ExitDecision(
                    "full_sell",
                    "ema_9_21_death_cross",
                    total_qty,
                )

    # ══════════════════════════════════════════════════════════════════════
    # PRIORITY 6 — Stale trade (going nowhere for too long)
    # ══════════════════════════════════════════════════════════════════════
    if check_stale_trade(trade, pnl_pct):
        return ExitDecision("full_sell", "stale_trade_timeout", total_qty)

    # ══════════════════════════════════════════════════════════════════════
    # DEFAULT — Hold
    # ══════════════════════════════════════════════════════════════════════
    trail_pct = get_trailing_stop_pct(pnl_pct)
    log.info(
        f"{symbol}: HOLD | trail={trail_pct * 100:.0f}% "
        f"stop=${stop_level:.4f} | phase={phase}"
    )
    return ExitDecision("hold", "holding", 0)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN MONITOR LOOP
# ══════════════════════════════════════════════════════════════════════════════

def monitor_positions(ibkr: IBKRSellClient):
    log.info("=" * 60)
    log.info("SELL BOT v2.3 — Smart Position Manager")
    log.info("  Liquidity crisis: DISABLED (removed)")
    log.info("  Exit confirmation: PSAR bearish only")
    log.info("  Duplicate positions: auto-consolidated")
    log.info(f"Hard Stop: {HARD_STOP_PCT * 100}% | "
             f"Partial: {PARTIAL_PROFIT_PCT * 100}% ({PARTIAL_SELL_FRACTION * 100}% of qty) | "
             f"Flash: {FLASH_CRASH_PCT * 100}%")
    log.info(f"Trailing tiers: {[(f'+{t*100:.0f}%', f'{p*100:.0f}%') for t, p in TRAILING_TIERS]}")
    log.info(f"Trail confirmation: PSAR bearish ONLY")
    log.info(f"Re-entry cooldown: {REENTRY_COOLDOWN_MIN} min")
    log.info(f"Stale exit: >{STALE_TRADE_HOURS}h if PnL in "
             f"[{STALE_TRADE_MIN_PNL * 100}%, {STALE_TRADE_MAX_PNL * 100}%]")
    log.info("=" * 60)

    while True:
        try:
            open_trades = get_open_trades()

            # ── Consolidate duplicate positions before processing ─────────
            open_trades = consolidate_duplicate_positions(open_trades)

            log.info(f"--- Checking {len(open_trades)} open position(s) ---")

            for trade in open_trades:
                symbol    = trade.get("instrument") or trade.get("symbol")
                total_qty = trade.get("quantity") or trade.get("qty")

                if not symbol or not total_qty:
                    log.warning(f"Skipping trade {trade['_id']}: missing symbol or qty")
                    continue

                entry_price   = get_effective_entry_price(trade, ibkr)
                current_price = ibkr.get_last_price(symbol)
                if not current_price:
                    log.warning(f"{symbol}: no current price, skipping")
                    continue

                decision = evaluate_position(trade, current_price, entry_price, ibkr)

                if decision.action == "hold":
                    continue

                # ── Execute the decision ──────────────────────────────────

                if decision.action == "emergency_sell":
                    exit_price, final_reason = emergency_exit(
                        ibkr, symbol, decision.qty_to_sell, decision.reason
                    )
                    close_trade(trade, exit_price, final_reason, decision.qty_to_sell)

                elif decision.action == "partial_sell":
                    sell_trade    = ibkr.place_limit_sell(symbol, current_price,
                                                          decision.qty_to_sell)
                    sell_order_id = sell_trade.order.orderId
                    confirmed     = ibkr.get_sell_fill_price(symbol, sell_order_id,
                                                               current_price)
                    close_trade(trade, confirmed, decision.reason, decision.qty_to_sell)

                elif decision.action == "full_sell":
                    sell_trade    = ibkr.place_limit_sell(symbol, current_price,
                                                          decision.qty_to_sell)
                    sell_order_id = sell_trade.order.orderId
                    confirmed     = ibkr.get_sell_fill_price(symbol, sell_order_id,
                                                               current_price)
                    close_trade(trade, confirmed, decision.reason, decision.qty_to_sell)

        except Exception as e:
            log.error(f"Monitor error: {e}", exc_info=True)

        time.sleep(MONITOR_INTERVAL)


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

def main():
    log.info("=== Sell Bot v2.3 Starting ===")
    ibkr = IBKRSellClient()

    try:
        ibkr.connect()
        monitor_positions(ibkr)
    except KeyboardInterrupt:
        log.info("Sell bot stopped by user.")
    except Exception as e:
        log.error(f"Fatal error: {e}", exc_info=True)
    finally:
        ibkr.disconnect()
        log.info("IBKR disconnected.")


if __name__ == "__main__":
    main()

